# ARC-AGI-3 Duck v12 — Qwen3.8 ADLDB-DWE — LOAD ALL INPUTS + MODEL

**Required model:** `Qwen/Qwen3.8-27B-FP8`  
**Seed:** `20260819`  
**Action caps:** `ls20 = 309`; every other game action-uncapped by DWE.

This notebook has a hard startup gate. Before any game environment can be opened it must:

1. resolve and open the ARC-AGI-3 competition input;
2. resolve and open the TAAF source bundle;
3. resolve and open the ARC3 vLLM wheelhouse;
4. resolve and open a complete Qwen3.8-27B-FP8 snapshot;
5. load all bundled source trees into `PYTHONPATH`;
6. install/reconcile all offline requirements;
7. start vLLM against the resolved Qwen3.8 files;
8. verify `/v1/models`;
9. send a real chat-completion request and receive a non-empty response;
10. write a final `/kaggle/working/INPUT_MODEL_READY.json` gate file.

**Gameplay cannot start unless that gate file exists and all checks are `true`.**


In [ ]:
# === STAGE 1: INVENTORY + RESOLVE EVERY REQUIRED KAGGLE INPUT ===
from pathlib import Path
import json
import os

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_UTILITY_ROOT = Path("/kaggle/usr/lib")
KAGGLE_MODEL_ROOT = Path("/kaggle/models")
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUT_CLASSES = {
    "competition": "ARC Prize 2026 - ARC-AGI-3",
    "taaf_source": "TAAF Kaggle Source Bundle",
    "vllm_wheelhouse": "ARC3 vLLM H100 Wheelhouse V3",
    "qwen38_model": "Qwen3.8-27B-FP8",
}

print("=" * 100)
print("STAGE 1 — KAGGLE MOUNT INVENTORY")
for root in (KAGGLE_INPUT_ROOT, KAGGLE_UTILITY_ROOT, KAGGLE_MODEL_ROOT):
    print(f"ROOT {root} exists={root.exists()}")
    if root.exists():
        try:
            children = sorted(root.iterdir(), key=lambda p: p.name.lower())
        except OSError:
            children = []
        for child in children[:200]:
            print(f"  {child}")
print("=" * 100)


def _first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)


def _find_any(root: Path, name: str):
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require_path(path, label):
    if path is None:
        raise FileNotFoundError(f"REQUIRED INPUT MISSING: {label}")
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"REQUIRED INPUT MISSING: {label}: {path}")
    return path


# Competition
ARC_COMPETITION_ROOT = _first_existing([
    KAGGLE_INPUT_ROOT / "arc-prize-2026-arc-agi-3",
    KAGGLE_INPUT_ROOT / "competitions" / "arc-prize-2026-arc-agi-3",
])
if ARC_COMPETITION_ROOT is None:
    _arc_wheels_hit = _find_any(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if _arc_wheels_hit is not None:
        ARC_COMPETITION_ROOT = _arc_wheels_hit.parent
ARC_COMPETITION_ROOT = _require_path(
    ARC_COMPETITION_ROOT,
    "ARC Prize 2026 - ARC-AGI-3 competition input",
)

ARC_WHEELS_DIR = _require_path(
    (
        ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
        if (ARC_COMPETITION_ROOT / "arc_agi_3_wheels").is_dir()
        else _find_any(ARC_COMPETITION_ROOT, "arc_agi_3_wheels")
    ),
    "ARC competition wheel directory",
)

ARC_ENVIRONMENTS_DIR = _require_path(
    (
        ARC_COMPETITION_ROOT / "environment_files"
        if (ARC_COMPETITION_ROOT / "environment_files").is_dir()
        else _find_any(ARC_COMPETITION_ROOT, "environment_files")
    ),
    "ARC environment_files",
)

# TAAF
_taaf_marker = _find_any(KAGGLE_INPUT_ROOT, "taaf-kaggle-bundle.json")
if _taaf_marker is None:
    raise FileNotFoundError(
        "REQUIRED INPUT MISSING: TAAF Kaggle Source Bundle "
        "(taaf-kaggle-bundle.json not found)"
    )
BUNDLE_DIR = _taaf_marker.parent.resolve()

for _name in (
    "taaf-kaggle-bundle.json",
    "setup_commands.json",
    "teardown_commands.json",
    "deploy_target.pkl",
    "benchmark_initial.pkl",
):
    _require_path(BUNDLE_DIR / _name, f"TAAF artifact {_name}")
_require_path(BUNDLE_DIR / "src", "TAAF src directory")

# vLLM wheelhouse
_vllm_lock = _find_any(KAGGLE_INPUT_ROOT, "requirements.lock")
if _vllm_lock is None:
    raise FileNotFoundError(
        "REQUIRED INPUT MISSING: ARC3 vLLM H100 Wheelhouse V3 "
        "(requirements.lock not found)"
    )
VLLM_WHEELHOUSE_DIR = _vllm_lock.parent.resolve()
_require_path(
    VLLM_WHEELHOUSE_DIR / "requirements.lock",
    "vLLM requirements.lock",
)

# Make source root explicit immediately. Individual bundled repos are added later.
os.environ["ARC_AGI3_COMPETITION_ROOT"] = str(ARC_COMPETITION_ROOT)
os.environ["ARC_AGI3_WHEELS_DIR"] = str(ARC_WHEELS_DIR)
os.environ["ARC_AGI3_ENVIRONMENTS_DIR"] = str(ARC_ENVIRONMENTS_DIR)
os.environ["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
os.environ["TAAF_VLLM_WHEELHOUSE"] = str(VLLM_WHEELHOUSE_DIR)

INPUT_STAGE1 = {
    "competition": str(ARC_COMPETITION_ROOT),
    "arc_wheels": str(ARC_WHEELS_DIR),
    "environment_files": str(ARC_ENVIRONMENTS_DIR),
    "taaf_source": str(BUNDLE_DIR),
    "vllm_wheelhouse": str(VLLM_WHEELHOUSE_DIR),
}

(WORKING_DIR / "stage1_inputs.json").write_text(
    json.dumps(INPUT_STAGE1, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=" * 100)
print("STAGE 1 — STATIC INPUTS RESOLVED")
for k, v in INPUT_STAGE1.items():
    print(f"{k:24s}: {v}")
print("=" * 100)


In [ ]:

BUILD_ID = "ADLDB-Q38-FINAL-TESTED-v3-20260822"
print("=" * 100)
print(f"BUILD ID: {BUILD_ID}")
print("REQUIREMENTS POLICY: NO BULK SOURCE_MISSING PIP INSTALL")
print("SOURCE RESOLUTION: exact-installed -> importable -> vendored -> attached-wheel")
print("MODEL: Qwen/Qwen3.8-27B-FP8")
print("ACTION CAPS: ls20=309; all other games=UNCAPPED")
print("=" * 100)


import json
import os
import pickle
import random
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get(
    "KAGGLE_IS_COMPETITION_RERUN", ""
).strip().lower() in {"1", "true"}

NOTEBOOK_START_EPOCH = time.time()
CONTROL_SEED = int(os.environ.get("ADLDB_CONTROL_SEED", "20260819"))
KNOWN_PUBLIC_CONTROL_SEEDS = (20260819, 20260807)

ANALYZER_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
ANALYZER_CONTEXT_WINDOW = 32768
VLLM_MAX_MODEL_LEN = 65536

os.environ["PYTHONHASHSEED"] = str(CONTROL_SEED)
os.environ["ADLDB_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["TAAF_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["VLLM_SEED"] = str(CONTROL_SEED)

random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception:
    _np = None

try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception:
    _torch = None

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

os.environ["INFERENCE_ANALYZER_MODEL"] = ANALYZER_MODEL_ID
os.environ["LOCAL_ANALYZER_MODEL_ID"] = ANALYZER_MODEL_ID
os.environ["LOCAL_ANALYZER_BASE_URL"] = "http://127.0.0.1:1234/v1"
os.environ["TAAF_MAX_OUTPUT_TOKENS"] = "8192"
os.environ["TAAF_TOOL_STEPS"] = "8"
os.environ["TAAF_TEMPERATURE"] = "1.0"
os.environ["TAAF_TOP_P"] = "0.95"
os.environ["TAAF_TOP_K"] = "20"
os.environ["TAAF_CONTEXT_WINDOW"] = str(ANALYZER_CONTEXT_WINDOW)

os.environ["ARC3_FRAME_MODE"] = "full"
os.environ["ARC3_STATE_GRAPH"] = "off"
os.environ["ARC3_REEXPLORE_STRICT"] = "0"
os.environ["ADLDB_NO_IMPACT"] = "on"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [
        cuda_library_path,
        *os.environ.get("LIBRARY_PATH", "").split(os.pathsep),
    ]
    if entry
)

WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(
    "ADLDB CONTROL "
    f"seed={CONTROL_SEED} "
    f"model={ANALYZER_MODEL_ID} "
    f"context={ANALYZER_CONTEXT_WINDOW} "
    f"TRUE_SUBMISSION={TRUE_SUBMISSION}",
    flush=True,
)


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).


In [ ]:

# === RESOLVE ALL INPUTS, INCLUDING QWEN3.8 NOTEBOOK INPUT ===
from pathlib import Path
import json
import os
import re
import sys

REQUIRED_COMPETITION = "arc-prize-2026-arc-agi-3"
TAAF_SOURCE_REF = "jeroencottaar/taaf-kaggle-source-share"
VLLM_WHEELHOUSE_REF = "driessmit1/arc3-vllm-h100-wheelhouse-v3"

DATASET_SOURCES = [TAAF_SOURCE_REF, VLLM_WHEELHOUSE_REF]
KERNEL_SOURCES = ["qwen3.8-27B"]
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_UTILITY_ROOT = Path("/kaggle/usr/lib")


def _first_existing(paths):
    return next((p for p in paths if p.exists()), None)


def _find_named(root: Path, name: str):
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require(path, label):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"REQUIRED INPUT MISSING: {label}")
    return Path(path).resolve()


def _dataset_candidates(ref: str):
    owner, slug = ref.split("/", 1)
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "datasets" / owner / slug,
    ]


def _resolve_dataset(ref: str, marker: str):
    root = _first_existing(_dataset_candidates(ref))
    if root and _find_named(root, marker):
        return root.resolve()

    if KAGGLE_INPUT_ROOT.exists():
        marker_hit = _find_named(KAGGLE_INPUT_ROOT, marker)
        if marker_hit:
            return marker_hit.parent.resolve()

    raise FileNotFoundError(
        f"REQUIRED DATASET MISSING: {ref}; marker={marker}"
    )


def _has_model_weights(root: Path):
    if not root.exists():
        return False
    return (
        (root / "model.safetensors").exists()
        or (root / "model.safetensors.index.json").exists()
        or any(root.glob("*.safetensors"))
    )


def _read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _qwen38_score(model_dir: Path, cfg: dict):
    text = (
        str(model_dir)
        + " "
        + json.dumps(cfg, sort_keys=True, default=str)
    ).lower()

    score = 0
    indicators = (
        ("qwen3.8", 200),
        ("qwen3-8", 190),
        ("qwen38", 180),
        ("27b", 30),
        ("fp8", 30),
    )
    for token, points in indicators:
        if token in text:
            score += points

    qcfg = json.dumps(
        cfg.get("quantization_config", {}),
        sort_keys=True,
        default=str,
    ).lower()
    if "fp8" in qcfg or "float8" in qcfg:
        score += 20

    text_cfg = cfg.get("text_config", {})
    if isinstance(text_cfg, dict):
        if int(text_cfg.get("hidden_size", 0) or 0) == 5120:
            score += 5
        if int(text_cfg.get("num_hidden_layers", 0) or 0) == 64:
            score += 5

    return score


def _iter_model_configs():
    roots = [
        KAGGLE_INPUT_ROOT,
        KAGGLE_UTILITY_ROOT,
    ]
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        try:
            cfg_paths = list(root.rglob("config.json"))
        except OSError:
            cfg_paths = []

        for cfg_path in cfg_paths:
            model_dir = cfg_path.parent.resolve()
            if model_dir in seen:
                continue
            seen.add(model_dir)
            yield model_dir, cfg_path


def _utility_qwen_files():
    if not KAGGLE_UTILITY_ROOT.exists():
        return []

    hits = []
    try:
        for path in KAGGLE_UTILITY_ROOT.rglob("*"):
            if not path.is_file():
                continue
            lower = str(path).lower()
            if (
                "qwen3.8" in lower
                or "qwen3-8" in lower
                or "qwen38" in lower
                or "duck-qwen3-8" in lower
            ):
                hits.append(path)
    except OSError:
        pass
    return hits


def _candidate_paths_from_utility_source():
    """
    Statically inspect installed utility/notebook Python files for absolute
    Kaggle model paths. Do not execute arbitrary utility code.
    """
    candidates = []
    path_pattern = re.compile(
        r'["\'](/kaggle/(?:input|models|working)/[^"\']+)["\']'
    )

    for file_path in _utility_qwen_files():
        if file_path.suffix.lower() not in {".py", ".txt", ".json", ".sh"}:
            continue
        try:
            text = file_path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue

        for match in path_pattern.findall(text):
            path = Path(match)
            # Model paths are often to a file inside the HF snapshot.
            if path.is_file():
                path = path.parent
            if path.exists():
                candidates.append(path.resolve())

    # Unique, preserving discovery order.
    out = []
    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        out.append(path)
    return out


def _model_dirs_below(root: Path):
    """
    Find complete HF model snapshot directories below root.
    Limited to config.json parents, avoiding full arbitrary filesystem hashing.
    """
    if not root.exists():
        return []

    out = []
    try:
        cfg_paths = list(root.rglob("config.json"))
    except OSError:
        cfg_paths = []

    for cfg_path in cfg_paths:
        model_dir = cfg_path.parent.resolve()
        if _has_model_weights(model_dir):
            out.append(model_dir)
    return out


def _resolve_qwen38_model():
    explicit = os.environ.get("ADLDB_QWEN38_MODEL_DIR", "").strip()
    if explicit:
        model_dir = Path(explicit).resolve()
        cfg_path = model_dir / "config.json"
        if not cfg_path.exists() or not _has_model_weights(model_dir):
            raise FileNotFoundError(
                f"ADLDB_QWEN38_MODEL_DIR is not a complete HF snapshot: {model_dir}"
            )
        cfg = _read_json(cfg_path)
        score = _qwen38_score(model_dir, cfg)
        if score < 180:
            raise RuntimeError(
                f"Explicit model path does not validate as Qwen3.8: {model_dir}"
            )
        return model_dir, f"explicit-env(score={score})"

    # Search every location Kaggle can use for attached data or utility scripts.
    search_roots = [
        KAGGLE_INPUT_ROOT,
        KAGGLE_UTILITY_ROOT,
        Path("/kaggle/models"),
    ]

    model_dirs = []
    for root in search_roots:
        model_dirs.extend(_model_dirs_below(root))

    # Installed utility code may contain the actual attached model path.
    utility_paths = _candidate_paths_from_utility_source()
    for utility_path in utility_paths:
        if (utility_path / "config.json").exists() and _has_model_weights(utility_path):
            model_dirs.append(utility_path)
        else:
            model_dirs.extend(_model_dirs_below(utility_path))

    candidates = []
    seen = set()
    for model_dir in model_dirs:
        model_dir = model_dir.resolve()
        if model_dir in seen:
            continue
        seen.add(model_dir)

        cfg_path = model_dir / "config.json"
        cfg = _read_json(cfg_path)
        score = _qwen38_score(model_dir, cfg)

        path_text = str(model_dir).lower()
        if "duck-qwen3-8-27b-fp8" in path_text:
            score += 150
        elif "qwen3.8-27b" in path_text or "qwen3-8-27b" in path_text:
            score += 100
        elif "qwen38" in path_text:
            score += 80

        if score >= 180:
            candidates.append((score, model_dir, cfg))

    if candidates:
        candidates.sort(
            key=lambda item: (item[0], -len(str(item[1]))),
            reverse=True,
        )
        score, model_dir, cfg = candidates[0]

        identity_text = (
            str(model_dir)
            + " "
            + json.dumps(cfg, sort_keys=True, default=str)
        ).lower()

        if "qwen3.6" in identity_text or "qwen3-6" in identity_text:
            raise RuntimeError(
                f"WRONG MODEL RESOLVED: Qwen3.6 candidate at {model_dir}"
            )

        return model_dir.resolve(), f"filesystem-scan(score={score})"

    # Important diagnostic: Kaggle may have attached the notebook as a utility
    # script, but utility code itself is not the same thing as model weights.
    utility_hits = _utility_qwen_files()

    utility_listing = "\n".join(
        f"  - {p}"
        for p in utility_hits[:50]
    ) or "  (none)"

    roots = "\n".join(
        f"  - {root} exists={root.exists()}"
        for root in search_roots
    )

    if utility_hits:
        raise FileNotFoundError(
            "Qwen3.8 NOTEBOOK/UTILITY INPUT IS INSTALLED, BUT NO HF MODEL "
            "WEIGHTS WERE FOUND.\n"
            "Kaggle utility/notebook code was detected under /kaggle/usr/lib, "
            "but there is no directory containing both config.json and "
            "model.safetensors(.index.json).\n\n"
            "SEARCH ROOTS:\n"
            + roots
            + "\n\nQWEN UTILITY FILES FOUND:\n"
            + utility_listing
            + "\n\nThis means the attached notebook is code-only (or its saved "
              "version did not export the model weights). Add the Qwen3.8 FP8 "
              "MODEL/DATASET input used by that notebook, not only the notebook "
              "utility itself."
        )

    raise FileNotFoundError(
        "Qwen3.8 was not found as a dataset, Kaggle model, or installed "
        "utility/notebook input.\nSEARCH ROOTS:\n"
        + roots
    )


# ARC competition.
ARC_COMPETITION_ROOT = _first_existing([
    KAGGLE_INPUT_ROOT / REQUIRED_COMPETITION,
    KAGGLE_INPUT_ROOT / "competitions" / REQUIRED_COMPETITION,
])

if ARC_COMPETITION_ROOT is None:
    wheels = _find_named(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if wheels and wheels.is_dir():
        ARC_COMPETITION_ROOT = wheels.parent

ARC_COMPETITION_ROOT = _require(
    ARC_COMPETITION_ROOT,
    "ARC-AGI-3 competition",
)

ARC_WHEELS_DIR = ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
if not ARC_WHEELS_DIR.is_dir():
    ARC_WHEELS_DIR = _require(
        _find_named(ARC_COMPETITION_ROOT, "arc_agi_3_wheels"),
        "arc_agi_3_wheels",
    )
ARC_WHEELS_DIR = ARC_WHEELS_DIR.resolve()

ARC_ENVIRONMENTS_DIR = (
    ARC_COMPETITION_ROOT / "environment_files"
).resolve()

if not TRUE_SUBMISSION:
    ARC_ENVIRONMENTS_DIR = _require(
        ARC_ENVIRONMENTS_DIR if ARC_ENVIRONMENTS_DIR.is_dir() else None,
        "ARC environment_files",
    )

# TAAF.
taaf_mount = _resolve_dataset(
    TAAF_SOURCE_REF,
    DATASET_BUNDLE_MARKER,
)
bundle_marker = _find_named(
    taaf_mount,
    DATASET_BUNDLE_MARKER,
)
BUNDLE_DIR = _require(
    bundle_marker.parent if bundle_marker else None,
    "TAAF source bundle",
)

for required in (
    "src",
    "setup_commands.json",
    "teardown_commands.json",
    "deploy_target.pkl",
    "benchmark_initial.pkl",
):
    _require(
        BUNDLE_DIR / required,
        f"TAAF component {required}",
    )

# vLLM wheelhouse.
wheel_mount = _resolve_dataset(
    VLLM_WHEELHOUSE_REF,
    "requirements.lock",
)
wheel_lock = _find_named(
    wheel_mount,
    "requirements.lock",
)
VLLM_WHEELHOUSE_DIR = _require(
    wheel_lock.parent if wheel_lock else None,
    "vLLM wheelhouse",
)

# Qwen3.8 — dataset OR notebook input.
QWEN_MODEL_DIR, QWEN_MODEL_SOURCE = _resolve_qwen38_model()


def _validate_qwen38_snapshot(model_dir: Path):
    model_dir = model_dir.resolve()
    cfg_path = model_dir / 'config.json'
    tok_cfg = model_dir / 'tokenizer_config.json'
    if not cfg_path.is_file():
        raise FileNotFoundError(f'Qwen3.8 config.json missing: {cfg_path}')
    if not tok_cfg.is_file():
        raise FileNotFoundError(f'Qwen3.8 tokenizer_config.json missing: {tok_cfg}')
    if not ((model_dir / 'tokenizer.json').is_file() or (model_dir / 'vocab.json').is_file()):
        raise FileNotFoundError(f'Qwen3.8 tokenizer payload missing under {model_dir}')

    index_path = model_dir / 'model.safetensors.index.json'
    if index_path.is_file():
        index_payload = _read_json(index_path)
        weight_map = index_payload.get('weight_map', {})
        if not isinstance(weight_map, dict) or not weight_map:
            raise RuntimeError(f'Invalid safetensors index: {index_path}')
        shard_paths = [model_dir / n for n in sorted(set(str(v) for v in weight_map.values()))]
        missing = [str(p) for p in shard_paths if not p.is_file()]
        if missing:
            raise FileNotFoundError('Missing Qwen3.8 shards: ' + ', '.join(missing[:20]))
    else:
        shard_paths = sorted(model_dir.glob('*.safetensors'))
        if not shard_paths:
            raise FileNotFoundError(f'No Qwen3.8 safetensors weights found under {model_dir}')
    zero_size = [str(p) for p in shard_paths if p.stat().st_size <= 0]
    if zero_size:
        raise RuntimeError('Zero-byte Qwen3.8 shards: ' + ', '.join(zero_size[:20]))
    total_bytes = sum(p.stat().st_size for p in shard_paths)
    if total_bytes < 10 * 1024**3:
        raise RuntimeError(f'Qwen3.8 payload unexpectedly small: {total_bytes / 1024**3:.2f} GiB')

    cfg = _read_json(cfg_path)
    identity = (str(model_dir) + ' ' + json.dumps(cfg, sort_keys=True, default=str)).lower()
    if 'qwen3.6' in identity or 'qwen3-6' in identity:
        raise RuntimeError(f'Wrong Qwen generation resolved: {model_dir}')
    return {
        'model_dir': str(model_dir),
        'model_type': cfg.get('model_type'),
        'architectures': cfg.get('architectures'),
        'weight_shards': len(shard_paths),
        'weight_gib': round(total_bytes / 1024**3, 3),
    }

QWEN_SNAPSHOT_AUDIT = _validate_qwen38_snapshot(QWEN_MODEL_DIR)
print('PIPELINE STAGE 2 — QWEN3.8 SNAPSHOT VALIDATED')
print(json.dumps(QWEN_SNAPSHOT_AUDIT, indent=2, sort_keys=True, default=str))

print("=" * 92)
print("QWEN3.8 MODEL RESOLVED")
print(f"path      : {QWEN_MODEL_DIR}")
print(f"source    : {QWEN_MODEL_SOURCE}")
print(f"served id : {ANALYZER_MODEL_ID}")
print("=" * 92)

# Immutable TAAF setup script still contains its old Qwen3.6 dataset key.
# Alias ONLY its lookup key to the verified Qwen3.8 path.
LEGACY_TAAF_MODEL_REF = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"

kaggle_input_paths = {
    TAAF_SOURCE_REF: str(BUNDLE_DIR),
    VLLM_WHEELHOUSE_REF: str(VLLM_WHEELHOUSE_DIR),
    LEGACY_TAAF_MODEL_REF: str(QWEN_MODEL_DIR),
    ANALYZER_MODEL_ID: str(QWEN_MODEL_DIR),
    "qwen3.8-27B": str(QWEN_MODEL_DIR),
    f"competition:{REQUIRED_COMPETITION}": str(ARC_COMPETITION_ROOT),
}

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(
        kaggle_input_paths,
        sort_keys=True,
    ),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "ARC_AGI3_COMPETITION_ROOT": str(ARC_COMPETITION_ROOT),
    "ARC_AGI3_WHEELS_DIR": str(ARC_WHEELS_DIR),
    "ARC_AGI3_ENVIRONMENTS_DIR": str(ARC_ENVIRONMENTS_DIR),
    "TAAF_VLLM_WHEELHOUSE": str(VLLM_WHEELHOUSE_DIR),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
}

os.environ.update(setup_env)

SETUP_ENV_PATH.write_text(
    json.dumps(
        setup_env,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

INPUT_MANIFEST = {
    "competition": str(ARC_COMPETITION_ROOT),
    "arc_wheels": str(ARC_WHEELS_DIR),
    "environment_files": str(ARC_ENVIRONMENTS_DIR),
    "taaf_source": str(BUNDLE_DIR),
    "vllm_wheelhouse": str(VLLM_WHEELHOUSE_DIR),
    "qwen38_model": str(QWEN_MODEL_DIR),
    "qwen38_resolution": QWEN_MODEL_SOURCE,
    "served_model_id": ANALYZER_MODEL_ID,
    "control_seed": CONTROL_SEED,
    "qwen_snapshot_audit": QWEN_SNAPSHOT_AUDIT,
}

(WORKING_DIR / "adldb_input_manifest.json").write_text(
    json.dumps(
        INPUT_MANIFEST,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

print("=" * 92)
print("ADLDB REQUIRED INPUTS — ALL RESOLVED")
for key, value in INPUT_MANIFEST.items():
    print(f"{key:24s}: {value}")
print("=" * 92)


# === STAGE 2B: CANONICAL RESOLVED INPUT MANIFEST ===
# Every downstream stage consumes this one manifest. No later cell is allowed
# to guess a new path independently.
RESOLVED_INPUTS = {
    "competition": str(ARC_COMPETITION_ROOT.resolve()),
    "arc_wheels": str(ARC_WHEELS_DIR.resolve()),
    "environment_files": str(ARC_ENVIRONMENTS_DIR.resolve()),
    "taaf_source": str(BUNDLE_DIR.resolve()),
    "vllm_wheelhouse": str(VLLM_WHEELHOUSE_DIR.resolve()),
    "qwen38_model": str(QWEN_MODEL_DIR.resolve()),
    "qwen38_source": str(QWEN_MODEL_SOURCE),
    "served_model_id": ANALYZER_MODEL_ID,
}

for _key, _value in RESOLVED_INPUTS.items():
    if _key in {"qwen38_source", "served_model_id"}:
        continue
    _path = Path(_value)
    if not _path.exists():
        raise FileNotFoundError(
            f"Resolved input disappeared before startup: {_key}={_path}"
        )

RESOLVED_INPUTS_PATH = WORKING_DIR / "resolved_inputs.json"
RESOLVED_INPUTS_PATH.write_text(
    json.dumps(
        RESOLVED_INPUTS,
        indent=2,
        sort_keys=True,
        default=str,
    ) + "\n",
    encoding="utf-8",
)

print("=" * 100)
print("STAGE 2B — ALL FOUR INPUT CLASSES RESOLVED")
for _key, _value in RESOLVED_INPUTS.items():
    print(f"{_key:24s}: {_value}")
print(f"manifest                : {RESOLVED_INPUTS_PATH}")
print("=" * 100)


In [ ]:
# === STAGE 3A: INSTALL + VERIFY ARC RUNTIME ===
# Offline only; no PyPI/network fallback is permitted.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(ARC_WHEELS_DIR),
        "arc-agi",
    ],
)

import importlib
import importlib.metadata as _metadata

_arc_module = importlib.import_module("arc_agi")
try:
    _arc_dist_version = _metadata.version("arc-agi")
except _metadata.PackageNotFoundError:
    _arc_dist_version = "source/unknown"

print(
    "PIPELINE STAGE 3A — ARC RUNTIME READY "
    f"module={getattr(_arc_module, '__file__', None)} "
    f"version={_arc_dist_version}",
    flush=True,
)


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.


In [ ]:

# === PUBLISH ALREADY-RESOLVED KAGGLE INPUTS TO TAAF ===
# The resolver above has already found and validated QWEN_MODEL_DIR.
# Do not perform a second mount-resolution pass here.

# Preserve all dataset/competition aliases created by the resolver.
kaggle_input_paths[ANALYZER_MODEL_ID] = str(QWEN_MODEL_DIR)

# The attached Kaggle utility/notebook is visible in the UI as qwen3.8-27B.
# TAAF only needs a stable logical name -> resolved path mapping.
for _logical_qwen_name in (
    "qwen3.8-27B",
    "duck-qwen3-8-27b-fp8",
    "Qwen/Qwen3.8-27B-FP8",
):
    kaggle_input_paths[_logical_qwen_name] = str(QWEN_MODEL_DIR)

# Keep the legacy lookup key because the immutable Duck/TAAF setup command
# still contains it. It is an alias ONLY; it points to the verified Qwen3.8
# directory and does not select or load Qwen3.6.
kaggle_input_paths[
    "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"
] = str(QWEN_MODEL_DIR)

KERNEL_SOURCES = [
    "qwen3.8-27B",
]

setup_env.update(
    {
        "TAAF_KAGGLE_INPUT_PATHS": json.dumps(
            kaggle_input_paths,
            sort_keys=True,
        ),
        "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(
            DATASET_SOURCES
        ),
        "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(
            KERNEL_SOURCES
        ),
        "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
        "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
    }
)

os.environ.update(setup_env)

SETUP_ENV_PATH.write_text(
    json.dumps(
        setup_env,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

print("=" * 92)
print("TAAF INPUT MAP PUBLISHED")
print(f"TAAF source     : {BUNDLE_DIR}")
print(f"Qwen3.8 model  : {QWEN_MODEL_DIR}")
print(f"served model id: {ANALYZER_MODEL_ID}")
print(
    "logical Qwen aliases:",
    [
        "qwen3.8-27B",
        "duck-qwen3-8-27b-fp8",
        "Qwen/Qwen3.8-27B-FP8",
    ],
)
print(
    "TAAF input paths:",
    setup_env["TAAF_KAGGLE_INPUT_PATHS"],
)
print("=" * 92)


_legacy_key = 'driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot'
if Path(kaggle_input_paths[_legacy_key]).resolve() != QWEN_MODEL_DIR.resolve():
    raise RuntimeError('Legacy TAAF model alias does not point to Qwen3.8.')
if Path(kaggle_input_paths[ANALYZER_MODEL_ID]).resolve() != QWEN_MODEL_DIR.resolve():
    raise RuntimeError('Analyzer model alias does not point to Qwen3.8.')
print('PIPELINE STAGE 4 — TAAF INPUT MAP VALIDATED', flush=True)


# === STAGE 4B: LOAD EVERY BUNDLED SOURCE TREE ===
SOURCE_ROOT = BUNDLE_DIR / "src"
SOURCE_PROJECT_PATHS = []

for _repo in sorted(
    [p for p in SOURCE_ROOT.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower(),
):
    for _candidate in (_repo / "src", _repo):
        if not _candidate.is_dir():
            continue
        _entry = str(_candidate.resolve())
        if _entry not in SOURCE_PROJECT_PATHS:
            SOURCE_PROJECT_PATHS.append(_entry)
        if _entry not in sys.path:
            sys.path.insert(0, _entry)

_existing_pp = [
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p
]
for _entry in reversed(SOURCE_PROJECT_PATHS):
    if _entry not in _existing_pp:
        _existing_pp.insert(0, _entry)
os.environ["PYTHONPATH"] = os.pathsep.join(_existing_pp)

if not SOURCE_PROJECT_PATHS:
    raise RuntimeError(f"No source projects found under {SOURCE_ROOT}")

SOURCE_PATHS_FILE = WORKING_DIR / "taaf_source_paths.json"
SOURCE_PATHS_FILE.write_text(
    json.dumps(
        {
            "source_root": str(SOURCE_ROOT),
            "python_paths": SOURCE_PROJECT_PATHS,
        },
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

print(
    "STAGE 4B — ALL BUNDLED SOURCE TREES LOADED "
    f"paths={len(SOURCE_PROJECT_PATHS)}",
    flush=True,
)
for _entry in SOURCE_PROJECT_PATHS:
    print(f"  PYTHONPATH { _entry }")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.


In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.

env = _command_env()

def _rewrite_setup_command(command: str) -> str:
    patched = str(command)
    # The immutable setup keeps its historical dataset key. The published
    # TAAF_KAGGLE_INPUT_PATHS mapping resolves that key to verified QWEN_MODEL_DIR.
    patched = patched.replace('vrfai/Qwen3.6-27B-FP8', ANALYZER_MODEL_ID)

    # Insert a real two-argument --seed entry into the nested Python argv list.
    if "'--seed'," not in patched and '"--seed",' not in patched:
        old = (
            "        '--max-model-len',\n"
            "        str(VLLM_MAX_MODEL_LEN),"
        )
        new = (
            "        '--seed',\n"
            "        os.getenv('VLLM_SEED', '20260819'),\n"
            "        '--max-model-len',\n"
            "        str(VLLM_MAX_MODEL_LEN),"
        )
        if old not in patched:
            raise RuntimeError('Could not safely locate vLLM --max-model-len argv block for seed insertion.')
        patched = patched.replace(old, new, 1)
    return patched

for raw_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    command = _rewrite_setup_command(raw_command)
    if command != raw_command:
        print(
            "taaf.kaggle: setup command model path upgraded "
            f"from legacy Qwen3.6 reference to {str(QWEN_MODEL_DIR)}",
            flush=True,
        )
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)


# === STAGE 5A: LOAD + RECONCILE ALL PYTHON REQUIREMENTS ===
# The immutable TAAF setup command has just installed the complete pinned
# vLLM requirements.lock into this target. Reconcile every pin, then load any
# additional dependencies declared by the bundled Duck/TAAF source projects.
import ast as _ast
import configparser as _configparser
import importlib as _importlib
import importlib.metadata as _metadata
import tempfile as _tempfile
import tomllib as _tomllib

VLLM_SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"
REQUIREMENTS_LOCK = VLLM_WHEELHOUSE_DIR / "requirements.lock"
REQUIREMENTS_AUDIT_PATH = WORKING_DIR / "all_requirements_audit.json"

if not VLLM_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(
        f"vLLM target site-packages was not created: {VLLM_SITE_PACKAGES}"
    )
if not REQUIREMENTS_LOCK.is_file():
    raise FileNotFoundError(
        f"Pinned vLLM requirements.lock missing: {REQUIREMENTS_LOCK}"
    )

# Make the isolated target visible to this notebook process before validating it.
if str(VLLM_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VLLM_SITE_PACKAGES))

_existing_pythonpath = [
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p
]
if str(VLLM_SITE_PACKAGES) not in _existing_pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(VLLM_SITE_PACKAGES), *_existing_pythonpath]
    )


def _canonical_dist_name(name: str) -> str:
    return re.sub(r"[-_.]+", "-", str(name).strip()).lower()


def _all_target_distributions():
    found = {}
    for dist in _metadata.distributions(path=[str(VLLM_SITE_PACKAGES)]):
        name = dist.metadata.get("Name")
        if name:
            found[_canonical_dist_name(name)] = str(dist.version)
    return found


def _all_visible_distributions():
    # Global/Kaggle environment first, isolated vLLM target overrides it.
    found = {}
    for dist in _metadata.distributions():
        name = dist.metadata.get("Name")
        if name:
            found[_canonical_dist_name(name)] = str(dist.version)
    found.update(_all_target_distributions())
    return found


try:
    from packaging.requirements import Requirement as _Requirement
    from packaging.markers import default_environment as _marker_environment
except Exception as _packaging_exc:
    raise RuntimeError(
        "The pinned runtime did not load the 'packaging' requirement."
    ) from _packaging_exc


def _parse_requirement_specs(lines):
    parsed = []
    for raw in lines:
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith(("-r ", "--requirement ", "-c ", "--constraint ")):
            # Nested requirement files are handled separately by pip when present;
            # the canonical wheelhouse lock currently contains direct pins.
            parsed.append({"raw": line, "nested": True})
            continue
        try:
            req = _Requirement(line)
        except Exception:
            parsed.append({"raw": line, "unparsed": True})
            continue
        marker_ok = True
        if req.marker is not None:
            marker_ok = bool(req.marker.evaluate(_marker_environment()))
        parsed.append(
            {
                "raw": line,
                "name": req.name,
                "canonical_name": _canonical_dist_name(req.name),
                "specifier": str(req.specifier),
                "marker": str(req.marker) if req.marker else None,
                "marker_applies": marker_ok,
            }
        )
    return parsed


LOCK_LINES = REQUIREMENTS_LOCK.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()
LOCK_REQUIREMENTS = _parse_requirement_specs(LOCK_LINES)
TARGET_DISTS = _all_target_distributions()

_lock_missing = []
_lock_wrong_version = []
_lock_unparsed = []

for item in LOCK_REQUIREMENTS:
    if item.get("nested"):
        continue
    if item.get("unparsed"):
        _lock_unparsed.append(item["raw"])
        continue
    if not item.get("marker_applies", True):
        continue

    installed = TARGET_DISTS.get(item["canonical_name"])
    if installed is None:
        _lock_missing.append(item["raw"])
        continue

    req = _Requirement(item["raw"])
    if req.specifier and installed not in req.specifier:
        _lock_wrong_version.append(
            {
                "requirement": item["raw"],
                "installed": installed,
            }
        )

if _lock_unparsed:
    raise RuntimeError(
        "Unparsed entries in requirements.lock: "
        + ", ".join(_lock_unparsed[:20])
    )
if _lock_missing or _lock_wrong_version:
    raise RuntimeError(
        "Pinned vLLM requirements reconciliation failed. "
        f"missing={_lock_missing[:20]} wrong_versions={_lock_wrong_version[:20]}"
    )

print(
    "FULL REQUIREMENTS: pinned vLLM lock reconciled "
    f"entries={len(LOCK_REQUIREMENTS)} "
    f"target_distributions={len(TARGET_DISTS)}",
    flush=True,
)


# ----- Discover dependency declarations from all bundled source projects -----

def _source_projects():
    root = BUNDLE_DIR / "src"
    return sorted(
        [p for p in root.iterdir() if p.is_dir()],
        key=lambda p: p.name,
    )


def _requirements_from_text_file(path: Path):
    specs = []
    for raw in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        # Local/VCS/editable references are source-layout concerns rather than
        # external runtime distributions. Record them, but do not ask offline pip
        # to resolve a network URL.
        _lower_line = line.lower()
        _is_direct_url = (
            _lower_line.startswith(("-e ", "--editable ", "git+", "http://", "https://"))
            or " @ git+" in _lower_line
            or " @ http://" in _lower_line
            or " @ https://" in _lower_line
            or " @ file://" in _lower_line
        )
        if _is_direct_url:
            specs.append(("local_or_external_reference", line))
        elif line.startswith(("-r ", "--requirement ")):
            nested = line.split(maxsplit=1)[1].strip()
            nested_path = (path.parent / nested).resolve()
            if not nested_path.is_file():
                raise FileNotFoundError(
                    f"Nested requirements file missing: {nested_path}"
                )
            specs.extend(_requirements_from_text_file(nested_path))
        elif line.startswith(("-c ", "--constraint ")):
            # Constraints are validated by pip if an install is required.
            specs.append(("constraint", line))
        elif line.startswith("--"):
            # Index/options are not allowed because this notebook is offline-only.
            specs.append(("pip_option", line))
        else:
            specs.append(("requirement", line))
    return specs


def _requirements_from_pyproject(path: Path):
    try:
        payload = _tomllib.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        raise RuntimeError(f"Failed to parse {path}: {exc}") from exc

    project = payload.get("project", {})
    deps = project.get("dependencies", [])
    if deps is None:
        deps = []
    if not isinstance(deps, list):
        raise RuntimeError(f"project.dependencies is not a list in {path}")
    out = []
    for x in deps:
        spec = str(x).strip()
        lower = spec.lower()
        is_direct = (
            lower.startswith(("git+", "http://", "https://"))
            or " @ git+" in lower
            or " @ http://" in lower
            or " @ https://" in lower
            or " @ file://" in lower
        )
        out.append(
            ("local_or_external_reference" if is_direct else "requirement", spec)
        )
    return out


def _requirements_from_setup_cfg(path: Path):
    parser = _configparser.ConfigParser()
    parser.read(path, encoding="utf-8")
    if not parser.has_option("options", "install_requires"):
        return []
    raw = parser.get("options", "install_requires")
    specs = []
    for line in raw.splitlines():
        line = line.strip()
        if line:
            lower = line.lower()
            is_direct = (
                lower.startswith(("git+", "http://", "https://"))
                or " @ git+" in lower
                or " @ http://" in lower
                or " @ https://" in lower
                or " @ file://" in lower
            )
            specs.append(
                ("local_or_external_reference" if is_direct else "requirement", line)
            )
    return specs


def _requirements_from_setup_py(path: Path):
    # Static literal-only extraction. Never execute arbitrary setup.py.
    try:
        tree = _ast.parse(path.read_text(encoding="utf-8", errors="replace"))
    except Exception:
        return []

    for node in _ast.walk(tree):
        if not isinstance(node, _ast.Call):
            continue
        fn_name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
        if fn_name != "setup":
            continue
        for kw in node.keywords:
            if kw.arg != "install_requires":
                continue
            try:
                value = _ast.literal_eval(kw.value)
            except Exception:
                return []
            if isinstance(value, (list, tuple)):
                out = []
                for item in value:
                    spec = str(item).strip()
                    lower = spec.lower()
                    is_direct = (
                        lower.startswith(("git+", "http://", "https://"))
                        or " @ git+" in lower
                        or " @ http://" in lower
                        or " @ https://" in lower
                        or " @ file://" in lower
                    )
                    out.append(
                        ("local_or_external_reference" if is_direct else "requirement", spec)
                    )
                return out
    return []


SOURCE_DECLARATIONS = []
SOURCE_PROJECTS = _source_projects()

for project in SOURCE_PROJECTS:
    for req_path in sorted(project.rglob("requirements*.txt")):
        SOURCE_DECLARATIONS.extend(
            (project.name, str(req_path), kind, spec)
            for kind, spec in _requirements_from_text_file(req_path)
        )

    pyproject = project / "pyproject.toml"
    if pyproject.is_file():
        SOURCE_DECLARATIONS.extend(
            (project.name, str(pyproject), kind, spec)
            for kind, spec in _requirements_from_pyproject(pyproject)
        )

    setup_cfg = project / "setup.cfg"
    if setup_cfg.is_file():
        SOURCE_DECLARATIONS.extend(
            (project.name, str(setup_cfg), kind, spec)
            for kind, spec in _requirements_from_setup_cfg(setup_cfg)
        )

    setup_py = project / "setup.py"
    if setup_py.is_file():
        SOURCE_DECLARATIONS.extend(
            (project.name, str(setup_py), kind, spec)
            for kind, spec in _requirements_from_setup_py(setup_py)
        )


# Resolve direct VCS/URL dependencies against the source bundle instead of
# attempting a network clone. Kaggle competition runs are offline.
def _canonical_project_token(value: str) -> str:
    value = str(value or "").strip()
    # PEP 508 direct reference: "name @ git+https://..."
    if " @ " in value:
        value = value.split(" @ ", 1)[0].strip()
    # Bare VCS URL: use repository basename.
    if "://" in value:
        value = value.rstrip("/").rsplit("/", 1)[-1]
    if value.endswith(".git"):
        value = value[:-4]
    return _canonical_dist_name(value)


BUNDLED_PROJECT_INDEX = {
    _canonical_project_token(p.name): p
    for p in SOURCE_PROJECTS
}

# Common naming aliases in the TAAF source family.
for _project in SOURCE_PROJECTS:
    _token = _canonical_project_token(_project.name)
    BUNDLED_PROJECT_INDEX.setdefault(_token.replace("tufal-", ""), _project)
    BUNDLED_PROJECT_INDEX.setdefault("tufal-" + _token, _project)


def _resolve_direct_reference_locally(spec: str):
    token = _canonical_project_token(spec)
    direct = BUNDLED_PROJECT_INDEX.get(token)
    if direct is not None:
        return direct

    # Match a requirement name to package metadata in vendored pyproject/setup.
    for project in SOURCE_PROJECTS:
        candidates = [project / "pyproject.toml", project / "setup.cfg", project / "setup.py"]
        joined = ""
        for candidate in candidates:
            if candidate.is_file():
                try:
                    joined += "\n" + candidate.read_text(
                        encoding="utf-8",
                        errors="ignore",
                    )
                except OSError:
                    pass
        if token and token in _canonical_project_token(joined):
            return project

    return None


LOCAL_DIRECT_REFERENCE_AUDIT = []

# Deduplicate external requirement specs while retaining an audit trail.
SOURCE_REQUIREMENT_SPECS = []
_seen_specs = set()
SOURCE_NONINSTALL_DECLARATIONS = []

for project_name, source_file, kind, spec in SOURCE_DECLARATIONS:
    if kind == "requirement":
        if spec not in _seen_specs:
            SOURCE_REQUIREMENT_SPECS.append(spec)
            _seen_specs.add(spec)
    else:
        _record = {
            "project": project_name,
            "source": source_file,
            "kind": kind,
            "value": spec,
        }

        if kind == "local_or_external_reference":
            _local_project = _resolve_direct_reference_locally(spec)
            _record["local_resolution"] = (
                str(_local_project) if _local_project is not None else None
            )

            if _local_project is None:
                raise RuntimeError(
                    "Offline direct dependency is not vendored in the TAAF bundle: "
                    f"{spec!r} declared by {source_file}. "
                    "Network cloning is intentionally disabled."
                )

            # Ensure this local project is importable by the notebook and every
            # vLLM/TAAF child process. We do not pip-install it or clone it.
            for _entry in (_local_project / "src", _local_project):
                if _entry.is_dir():
                    _entry_str = str(_entry.resolve())
                    if _entry_str not in sys.path:
                        sys.path.insert(0, _entry_str)

                    _pp = [
                        p
                        for p in os.environ.get("PYTHONPATH", "").split(os.pathsep)
                        if p
                    ]
                    if _entry_str not in _pp:
                        os.environ["PYTHONPATH"] = os.pathsep.join(
                            [_entry_str, *_pp]
                        )

            LOCAL_DIRECT_REFERENCE_AUDIT.append(
                {
                    "requirement": spec,
                    "resolved_to": str(_local_project.resolve()),
                    "network_fetch": False,
                }
            )

        SOURCE_NONINSTALL_DECLARATIONS.append(_record)


def _requirement_satisfied(spec: str, visible_versions: dict):
    req = _Requirement(spec)
    if req.marker is not None and not req.marker.evaluate(_marker_environment()):
        return True, "marker-not-applicable"

    installed = visible_versions.get(_canonical_dist_name(req.name))
    if installed is None:
        return False, None

    if req.specifier and installed not in req.specifier:
        return False, installed
    return True, installed


VISIBLE_DISTS = _all_visible_distributions()


# === BUILD / STALE-PATH GUARD ===
if BUILD_ID != "ADLDB-Q38-FINAL-TESTED-v3-20260822":
    raise RuntimeError(f"Wrong notebook build loaded: {BUILD_ID!r}")

print(
    "OFFLINE REQUIREMENTS RESOLVER ACTIVE — "
    "bulk SOURCE_MISSING pip path is DISABLED",
    flush=True,
)

# -------------------------------------------------------------------------
# Offline source-requirement resolver
#
# The vLLM requirements.lock remains strict and exact. Source-project
# requirements are different: some are development/diagnostic pins, some
# correspond to source already vendored in the TAAF bundle, and Kaggle's base
# image may already provide a working compatible runtime distribution.
#
# Resolution order:
#   1. exact declared version already installed -> exact;
#   2. package import works from current Kaggle/base environment -> runtime;
#   3. matching project exists in bundled TAAF source -> vendored source;
#   4. matching wheel exists anywhere in attached Kaggle inputs -> local wheel;
#   5. otherwise fail before gameplay.
#
# No network access and no VCS clone are ever attempted.
# -------------------------------------------------------------------------

def _distribution_to_import_candidates(dist_name: str):
    canonical = _canonical_dist_name(dist_name)
    explicit = {
        "matplotlib": ["matplotlib"],
        "requests": ["requests"],
        "kaggle": ["kaggle"],
        "tuf-arc-agi-framework": [
            "tuf_arc_agi_framework",
            "arc_agi",
            "inference",
        ],
        "tufa-arc-agi-framework": [
            "tufa_arc_agi_framework",
            "arc_agi",
            "inference",
        ],
        "arc-agi": ["arc_agi"],
        "pyarrow": ["pyarrow"],
        "numpy": ["numpy"],
        "pandas": ["pandas"],
        "torch": ["torch"],
        "packaging": ["packaging"],
    }
    if canonical in explicit:
        return explicit[canonical]

    # Standard Python package-name normalization is a useful fallback.
    return [
        canonical.replace("-", "_"),
        canonical.split("-", 1)[0].replace("-", "_"),
    ]


def _runtime_import_available(req):
    """
    Determine whether the declared dependency capability is already available.
    This deliberately checks importability in addition to distribution metadata,
    because Kaggle and the vendored TAAF bundle can expose modules without a
    matching wheel metadata record.
    """
    candidates = _distribution_to_import_candidates(req.name)
    successes = []
    failures = []

    for module_name in candidates:
        try:
            module = _importlib.import_module(module_name)
        except Exception as exc:
            failures.append(
                {
                    "module": module_name,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            continue

        successes.append(
            {
                "module": module_name,
                "file": str(getattr(module, "__file__", None)),
                "version": str(getattr(module, "__version__", None)),
            }
        )

    return bool(successes), successes, failures


def _matching_vendored_project(req):
    token = _canonical_dist_name(req.name)

    # Name-based match.
    for project in SOURCE_PROJECTS:
        ptoken = _canonical_dist_name(project.name)
        aliases = {
            ptoken,
            ptoken.replace("tufal-", ""),
            "tufal-" + ptoken,
        }
        if token in aliases:
            return project

    # Metadata-content match.
    for project in SOURCE_PROJECTS:
        metadata_files = [
            project / "pyproject.toml",
            project / "setup.cfg",
            project / "setup.py",
        ]
        for metadata_file in metadata_files:
            if not metadata_file.is_file():
                continue
            try:
                raw = metadata_file.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ).lower()
            except OSError:
                continue

            # Look for normalized requirement/project name in metadata.
            normalized_raw = re.sub(r"[-_.]+", "-", raw)
            if token in normalized_raw:
                return project

    return None


def _activate_vendored_project(project: Path):
    activated = []
    for candidate in (project / "src", project):
        if not candidate.is_dir():
            continue
        entry = str(candidate.resolve())
        if entry not in sys.path:
            sys.path.insert(0, entry)

        pythonpath = [
            p
            for p in os.environ.get("PYTHONPATH", "").split(os.pathsep)
            if p
        ]
        if entry not in pythonpath:
            os.environ["PYTHONPATH"] = os.pathsep.join(
                [entry, *pythonpath]
            )
        activated.append(entry)
    return activated


def _all_attached_wheel_roots():
    roots = [
        VLLM_WHEELHOUSE_DIR,
        ARC_WHEELS_DIR,
        KAGGLE_INPUT_ROOT,
        KAGGLE_UTILITY_ROOT,
        KAGGLE_MODEL_ROOT,
    ]
    result = []
    seen = set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        resolved = root.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        result.append(resolved)
    return result


def _local_wheels_for_requirement(req):
    """
    Find locally attached wheels for a distribution. We do not require a
    particular directory layout, since Kaggle datasets/notebook inputs can
    nest wheel files under arbitrary subdirectories.
    """
    canonical = _canonical_dist_name(req.name)
    prefixes = {
        canonical,
        canonical.replace("-", "_"),
    }

    hits = []
    seen = set()
    for root in _all_attached_wheel_roots():
        try:
            wheel_paths = root.rglob("*.whl")
        except OSError:
            continue

        for wheel in wheel_paths:
            filename = wheel.name.lower()
            normalized = filename.replace("_", "-")
            if not any(normalized.startswith(p + "-") for p in prefixes):
                continue
            resolved = wheel.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            hits.append(resolved)

    return sorted(hits, key=lambda p: p.name)


def _install_local_wheel_requirement(spec: str, wheel_paths):
    """
    Ask pip to resolve the declared spec only from directories containing
    matching wheels. This preserves version constraints when an exact local
    wheel exists, without ever consulting the network.
    """
    dirs = []
    seen = set()
    for wheel in wheel_paths:
        parent = wheel.parent.resolve()
        if parent not in seen:
            seen.add(parent)
            dirs.append(parent)

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--target",
        str(VLLM_SITE_PACKAGES),
        "--upgrade",
        "--only-binary",
        ":all:",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
    ]
    for directory in dirs:
        cmd.extend(["--find-links", str(directory)])
    cmd.append(spec)

    subprocess.check_call(cmd)


SOURCE_REQUIREMENT_AUDIT = []
SOURCE_UNRESOLVED = []

# Refresh after all source paths / vendored direct refs have been activated.
VISIBLE_DISTS = _all_visible_distributions()

for spec in SOURCE_REQUIREMENT_SPECS:
    try:
        req = _Requirement(spec)
    except Exception as exc:
        raise RuntimeError(
            f"Could not parse bundled source requirement {spec!r}: {exc}"
        ) from exc

    if req.marker is not None and not req.marker.evaluate(_marker_environment()):
        SOURCE_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "marker-not-applicable",
            }
        )
        continue

    canonical = _canonical_dist_name(req.name)
    installed_version = VISIBLE_DISTS.get(canonical)

    # Tier 1: exact distribution requirement is already satisfied.
    if (
        installed_version is not None
        and (not req.specifier or installed_version in req.specifier)
    ):
        SOURCE_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "exact-installed",
                "installed": installed_version,
            }
        )
        continue

    # Kaggle itself is a host/runtime capability. Some images expose the
    # notebook environment without a separately importable `kaggle` distribution.
    # Do not try to install Kaggle inside Kaggle.
    if canonical == "kaggle" and Path("/kaggle").exists():
        SOURCE_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "host-provided-kaggle-runtime",
                "installed_distribution_version": installed_version,
            }
        )
        continue

    # Tier 2: the required runtime capability is already importable.
    import_ok, import_successes, import_failures = _runtime_import_available(req)
    if import_ok:
        SOURCE_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "runtime-import-compatible",
                "declared_specifier": str(req.specifier),
                "installed_distribution_version": installed_version,
                "imports": import_successes,
                "note": (
                    "Exact source-project pin was not reinstalled because "
                    "the required runtime capability is already available "
                    "in Kaggle/bundled source. vLLM lock pins remain strict."
                ),
            }
        )
        continue

    # Tier 3: plain requirement maps to a vendored source project.
    vendored = _matching_vendored_project(req)
    if vendored is not None:
        activated = _activate_vendored_project(vendored)

        import_ok, import_successes, import_failures = _runtime_import_available(req)
        if import_ok:
            SOURCE_REQUIREMENT_AUDIT.append(
                {
                    "requirement": spec,
                    "status": "vendored-source",
                    "project": str(vendored.resolve()),
                    "python_paths": activated,
                    "imports": import_successes,
                }
            )
            continue

    # Tier 4: exact/compatible wheel anywhere in attached Kaggle inputs.
    local_wheels = _local_wheels_for_requirement(req)
    if local_wheels:
        try:
            _install_local_wheel_requirement(spec, local_wheels)
        except subprocess.CalledProcessError as exc:
            SOURCE_REQUIREMENT_AUDIT.append(
                {
                    "requirement": spec,
                    "status": "local-wheel-install-failed",
                    "wheels": [str(p) for p in local_wheels[:100]],
                    "error": str(exc),
                }
            )
        else:
            # Ensure isolated target is active.
            if str(VLLM_SITE_PACKAGES) not in sys.path:
                sys.path.insert(0, str(VLLM_SITE_PACKAGES))

            VISIBLE_DISTS = _all_visible_distributions()
            installed_version = VISIBLE_DISTS.get(canonical)
            exact_ok = (
                installed_version is not None
                and (not req.specifier or installed_version in req.specifier)
            )
            import_ok, import_successes, import_failures = _runtime_import_available(req)

            if exact_ok or import_ok:
                SOURCE_REQUIREMENT_AUDIT.append(
                    {
                        "requirement": spec,
                        "status": "installed-from-attached-wheel",
                        "installed": installed_version,
                        "wheels": [str(p) for p in local_wheels[:100]],
                        "imports": import_successes,
                    }
                )
                continue

    # Tier 5: genuinely unavailable.
    SOURCE_UNRESOLVED.append(
        {
            "requirement": spec,
            "installed_distribution_version": installed_version,
            "import_failures": import_failures,
            "vendored_project": (
                str(vendored.resolve()) if vendored is not None else None
            ),
            "local_wheels": [str(p) for p in local_wheels[:100]],
        }
    )

if SOURCE_UNRESOLVED:
    raise RuntimeError(
        "OFFLINE SOURCE REQUIREMENTS UNRESOLVED. "
        "The package is neither importable, vendored, nor available in any "
        f"attached local wheel: {SOURCE_UNRESOLVED}"
    )

SOURCE_MISSING = [
    item["requirement"]
    for item in SOURCE_REQUIREMENT_AUDIT
    if item["status"] == "installed-from-attached-wheel"
]

SOURCE_SATISFIED = [
    item
    for item in SOURCE_REQUIREMENT_AUDIT
    if item["status"] != "installed-from-attached-wheel"
]

print("=" * 96)
print("FULL REQUIREMENTS — SOURCE DEPENDENCIES RESOLVED OFFLINE")
for item in SOURCE_REQUIREMENT_AUDIT:
    print(
        f"  {item['requirement']} -> {item['status']}"
        + (
            f" ({item.get('installed')})"
            if item.get("installed") is not None
            else ""
        )
    )
print("network dependency fetches: 0")
print("=" * 96)


# ----- Explicit notebook/output dependencies -----
NOTEBOOK_REQUIREMENTS = [
    "numpy",
    "pandas",
    "pyarrow",
    "torch",
    "packaging",
]

NOTEBOOK_REQUIREMENT_AUDIT = []
NOTEBOOK_MISSING = []


def _notebook_module_name(spec: str) -> str:
    req = _Requirement(spec)
    return {
        "numpy": "numpy",
        "pandas": "pandas",
        "pyarrow": "pyarrow",
        "torch": "torch",
        "packaging": "packaging",
    }.get(_canonical_dist_name(req.name), req.name.replace("-", "_"))


for spec in NOTEBOOK_REQUIREMENTS:
    req = _Requirement(spec)
    canonical = _canonical_dist_name(req.name)
    visible = _all_visible_distributions()
    installed = visible.get(canonical)

    # First accept a distribution that satisfies the declared requirement.
    if installed is not None and (not req.specifier or installed in req.specifier):
        NOTEBOOK_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "installed",
                "version": installed,
            }
        )
        continue

    # Kaggle base images may provide the import even when metadata lives in an
    # unusual environment. Importability is the binding capability here.
    module_name = _notebook_module_name(spec)
    try:
        module = _importlib.import_module(module_name)
    except Exception as import_exc:
        module = None
        import_error = f"{type(import_exc).__name__}: {import_exc}"
    else:
        import_error = None

    if module is not None:
        NOTEBOOK_REQUIREMENT_AUDIT.append(
            {
                "requirement": spec,
                "status": "runtime-import-compatible",
                "module": module_name,
                "file": str(getattr(module, "__file__", None)),
                "version": str(getattr(module, "__version__", installed)),
            }
        )
        continue

    # Search ALL attached Kaggle wheel roots, not only the original two dirs.
    local_wheels = _local_wheels_for_requirement(req)
    if not local_wheels:
        raise RuntimeError(
            "REQUIRED NOTEBOOK PACKAGE UNAVAILABLE OFFLINE: "
            f"{spec}; import_error={import_error}; "
            "no matching wheel was found under any attached Kaggle input."
        )

    # Use the requirement spec, but disable dependency resolution so a one-off
    # notebook package cannot overwrite the already-reconciled vLLM lock.
    dirs = []
    seen_dirs = set()
    for wheel in local_wheels:
        parent = wheel.parent.resolve()
        if parent not in seen_dirs:
            seen_dirs.add(parent)
            dirs.append(parent)

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--target",
        str(VLLM_SITE_PACKAGES),
        "--upgrade",
        "--no-deps",
        "--only-binary",
        ":all:",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
    ]
    for directory in dirs:
        cmd.extend(["--find-links", str(directory)])
    cmd.append(spec)

    subprocess.check_call(cmd)

    if str(VLLM_SITE_PACKAGES) not in sys.path:
        sys.path.insert(0, str(VLLM_SITE_PACKAGES))
    _importlib.invalidate_caches()

    try:
        module = _importlib.import_module(module_name)
    except Exception as exc:
        raise RuntimeError(
            f"Notebook dependency {spec} installed from attached wheel "
            f"but import {module_name} still failed: {exc}"
        ) from exc

    NOTEBOOK_MISSING.append(spec)
    NOTEBOOK_REQUIREMENT_AUDIT.append(
        {
            "requirement": spec,
            "status": "installed-from-attached-wheel",
            "module": module_name,
            "wheels": [str(path) for path in local_wheels[:100]],
            "file": str(getattr(module, "__file__", None)),
            "version": str(getattr(module, "__version__", None)),
        }
    )


# ----- Import smoke tests for every layer used later in this notebook -----
REQUIRED_IMPORTS = [
    "arc_agi",
    "numpy",
    "pandas",
    "pyarrow",
    "torch",
    "packaging",
    "inference.agent.action_names",
    "inference.framework.solver",
    "inference.agent.tool_agent",
    "taaf.game_api",
]

IMPORT_AUDIT = {}
for module_name in REQUIRED_IMPORTS:
    try:
        module = _importlib.import_module(module_name)
    except Exception as exc:
        raise RuntimeError(
            f"Required runtime import failed: {module_name}: {exc}"
        ) from exc
    IMPORT_AUDIT[module_name] = {
        "file": str(getattr(module, "__file__", None)),
        "version": str(getattr(module, "__version__", None)),
    }

# Reconcile the pinned wheelhouse one final time after any additional installs.
FINAL_TARGET_DISTS = _all_target_distributions()
FINAL_LOCK_FAILURES = []
for item in LOCK_REQUIREMENTS:
    if item.get("nested") or item.get("unparsed"):
        continue
    if not item.get("marker_applies", True):
        continue
    installed = FINAL_TARGET_DISTS.get(item["canonical_name"])
    req = _Requirement(item["raw"])
    if installed is None or (req.specifier and installed not in req.specifier):
        FINAL_LOCK_FAILURES.append(
            {
                "requirement": item["raw"],
                "installed": installed,
            }
        )

if FINAL_LOCK_FAILURES:
    raise RuntimeError(
        f"Final wheelhouse lock reconciliation failed: {FINAL_LOCK_FAILURES[:30]}"
    )

ALL_REQUIREMENTS_AUDIT = {
    "requirements_lock": str(REQUIREMENTS_LOCK),
    "vllm_site_packages": str(VLLM_SITE_PACKAGES),
    "wheelhouse_lock_entries": len(LOCK_REQUIREMENTS),
    "wheelhouse_target_distributions": len(FINAL_TARGET_DISTS),
    "wheelhouse_lock_failures": FINAL_LOCK_FAILURES,
    "source_projects": [str(p) for p in SOURCE_PROJECTS],
    "source_requirement_specs": SOURCE_REQUIREMENT_SPECS,
    "source_requirements_initially_satisfied": SOURCE_SATISFIED,
    "source_requirements_installed": SOURCE_MISSING,
    "source_requirement_resolution": SOURCE_REQUIREMENT_AUDIT,
    "source_requirements_unresolved": SOURCE_UNRESOLVED,
    "source_noninstall_declarations": SOURCE_NONINSTALL_DECLARATIONS,
    "local_direct_references": LOCAL_DIRECT_REFERENCE_AUDIT,
    "network_dependency_fetches": 0,
    "notebook_requirements": NOTEBOOK_REQUIREMENTS,
    "notebook_requirement_resolution": NOTEBOOK_REQUIREMENT_AUDIT,
    "notebook_requirements_installed": NOTEBOOK_MISSING,
    "required_imports": IMPORT_AUDIT,
    "offline_only": True,
}

REQUIREMENTS_AUDIT_PATH.write_text(
    json.dumps(
        ALL_REQUIREMENTS_AUDIT,
        indent=2,
        sort_keys=True,
        default=str,
    ) + "\n",
    encoding="utf-8",
)

print("=" * 96)
print("PIPELINE STAGE 5A — ALL REQUIREMENTS LOADED")
print(
    f"vLLM lock entries      : {len(LOCK_REQUIREMENTS)}"
)
print(
    f"target distributions   : {len(FINAL_TARGET_DISTS)}"
)
print(
    f"source projects        : {len(SOURCE_PROJECTS)}"
)
print(
    f"source requirement specs: {len(SOURCE_REQUIREMENT_SPECS)}"
)
print(
    f"source deps installed  : {len(SOURCE_MISSING)}"
)
print(
    f"vendored direct refs   : {len(LOCAL_DIRECT_REFERENCE_AUDIT)}"
)
print("network dependency fetches: 0")
print(
    f"notebook deps installed: {len(NOTEBOOK_MISSING)}"
)
print(
    f"import smoke tests     : {len(IMPORT_AUDIT)}/{len(REQUIRED_IMPORTS)}"
)
print(
    f"audit                  : {REQUIREMENTS_AUDIT_PATH}"
)
print("=" * 96)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')

# Re-apply deterministic seeds after setup commands import/install numerical
# libraries. CUDA determinism is best-effort because vLLM may use kernels whose
# execution ordering is not bitwise deterministic.
random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception as _exc:
    print(f"taaf.kaggle: numpy seed warning: {_exc}", flush=True)

try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception as _exc:
    print(f"taaf.kaggle: torch seed warning: {_exc}", flush=True)

# Adopt the actual model ID exposed by the local vLLM server instead of assuming
# a served-model-name. This prevents dataset-directory names from breaking calls.
def _resolve_served_model_id(base_url: str, timeout_s: int = 180) -> str:
    from urllib.request import urlopen
    endpoint = base_url.rstrip("/") + "/models"
    deadline = time.monotonic() + timeout_s
    last_error = None
    while time.monotonic() < deadline:
        try:
            with urlopen(endpoint, timeout=10) as response:
                payload = json.loads(response.read().decode("utf-8"))
            models = payload.get("data", [])
            if models and models[0].get("id"):
                return str(models[0]["id"])
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(
        f"Local analyzer server did not expose a model at {endpoint}: {last_error}"
    )

_served_model_id = _resolve_served_model_id(
    os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
)

if _served_model_id != "Qwen/Qwen3.8-27B-FP8":
    raise RuntimeError(
        "WRONG MODEL SERVED: "
        f"expected=Qwen/Qwen3.8-27B-FP8 actual={_served_model_id}. "
        "Aborting before ARC gameplay."
    )

os.environ["INFERENCE_ANALYZER_MODEL"] = _served_model_id
os.environ["LOCAL_ANALYZER_MODEL_ID"] = _served_model_id
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(
    {
        "INFERENCE_ANALYZER_MODEL": _served_model_id,
        "LOCAL_ANALYZER_MODEL_ID": _served_model_id,
        "ARC3_CONTROL_SEED": str(CONTROL_SEED),
        "ARC3_FRAME_MODE": "full",
        "ARC3_STATE_GRAPH": "off",
    }
)
SETUP_ENV_PATH.write_text(
    json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(
    "CONTROL STACK READY "
    f"served_model={_served_model_id} seed={CONTROL_SEED} "
    f"frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"context={os.environ.get('TAAF_CONTEXT_WINDOW')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')}",
    flush=True,
)


if os.environ.get("INFERENCE_ANALYZER_MODEL") != "Qwen/Qwen3.8-27B-FP8":
    raise RuntimeError("Analyzer model identity was lost after setup.")
if not Path(os.environ["TAAF_QWEN_MODEL_DIR"]).resolve().exists():
    raise RuntimeError("Resolved Qwen3.8 model directory vanished after setup.")
print("PIPELINE STAGE 5 — VLLM + MODEL LOAD VALIDATED", flush=True)


# === STAGE 5B: REAL MODEL LOAD + CHAT COMPLETION HARD GATE ===
from urllib.request import Request as _Request
from urllib.request import urlopen as _urlopen

_MODEL_BASE = os.environ.get(
    "LOCAL_ANALYZER_BASE_URL",
    "http://127.0.0.1:1234/v1",
).rstrip("/")

# Confirm served identity once more from the live process.
with _urlopen(f"{_MODEL_BASE}/models", timeout=30) as _resp:
    _models_payload = json.loads(_resp.read().decode("utf-8"))

_live_ids = [
    str(item.get("id"))
    for item in (_models_payload.get("data") or [])
    if isinstance(item, dict) and item.get("id")
]
if ANALYZER_MODEL_ID not in _live_ids:
    raise RuntimeError(
        "LIVE MODEL IDENTITY FAILURE: "
        f"expected={ANALYZER_MODEL_ID} visible={_live_ids}"
    )

# Send an actual inference request. This proves model weights are loaded and the
# OpenAI-compatible endpoint can execute, not merely that a server process exists.
_smoke_payload = {
    "model": ANALYZER_MODEL_ID,
    "messages": [
        {
            "role": "user",
            "content": (
                "Reply with exactly READY. "
                "This is a startup health check, not an ARC game."
            ),
        }
    ],
    "temperature": 0.0,
    "max_tokens": 8,
}
_smoke_req = _Request(
    f"{_MODEL_BASE}/chat/completions",
    data=json.dumps(_smoke_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with _urlopen(_smoke_req, timeout=180) as _resp:
    _smoke_response = json.loads(_resp.read().decode("utf-8"))

try:
    _smoke_text = str(
        _smoke_response["choices"][0]["message"]["content"]
    ).strip()
except Exception as _exc:
    raise RuntimeError(
        f"Qwen3.8 chat-completion smoke response invalid: {_smoke_response}"
    ) from _exc

if not _smoke_text:
    raise RuntimeError(
        "Qwen3.8 chat-completion returned empty content."
    )

# Validate all earlier stage files still exist.
_REQUIRED_STAGE_FILES = [
    WORKING_DIR / "stage1_inputs.json",
    WORKING_DIR / "resolved_inputs.json",
    WORKING_DIR / "all_requirements_audit.json",
    WORKING_DIR / "taaf_source_paths.json",
]
_missing_stage_files = [
    str(p) for p in _REQUIRED_STAGE_FILES if not p.is_file()
]
if _missing_stage_files:
    raise RuntimeError(
        "Startup gate missing stage artifacts: "
        + ", ".join(_missing_stage_files)
    )

INPUT_MODEL_READY = {
    "all_static_inputs_loaded": True,
    "competition_loaded": True,
    "taaf_source_loaded": True,
    "vllm_wheelhouse_loaded": True,
    "all_requirements_loaded": True,
    "all_source_trees_loaded": True,
    "qwen38_snapshot_loaded": True,
    "vllm_server_started": True,
    "served_model_id": ANALYZER_MODEL_ID,
    "served_model_ids_visible": _live_ids,
    "chat_completion_smoke_passed": True,
    "chat_completion_text": _smoke_text,
    "resolved_inputs": RESOLVED_INPUTS,
}

INPUT_MODEL_READY_PATH = WORKING_DIR / "INPUT_MODEL_READY.json"
INPUT_MODEL_READY_PATH.write_text(
    json.dumps(
        INPUT_MODEL_READY,
        indent=2,
        sort_keys=True,
        default=str,
    ) + "\n",
    encoding="utf-8",
)

print("=" * 100)
print("INPUT + MODEL HARD GATE PASS")
print(f"served model : {ANALYZER_MODEL_ID}")
print(f"live ids     : {_live_ids}")
print(f"smoke output : {_smoke_text!r}")
print(f"gate file    : {INPUT_MODEL_READY_PATH}")
print("=" * 100)


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. ADLDB + Difference-Weighted Exploitation configuration

Every discovered game is still run once. `LS20_MAX_MOVES` remains the absolute safety ceiling, while DWE computes a **live per-game budget** from current-game evidence. A successful transition gets a protected exploit window; repeated no-progress, stalls, and loops reduce strategy weight and can trigger policy change or stop-loss.


In [ ]:
# === ADLDB / DIFFERENCE-WEIGHTED EXPLOITATION CONFIGURATION ===
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4
TARGET_SCORE_GAMES = 10
TARGET_MIN_GAME_SCORE = 0.40

# Binding action-cap contract: only ls20 is capped. All other games are
# action-uncapped by DWE; environment terminal state and wall-clock timeout remain.
LS20_MAX_MOVES = 309
GLOBAL_UNCAPPED_ACTION_LIMIT = 1_000_000_000
MAX_STALL_ACTIONS = 12
MAX_NO_PROGRESS_ACTIONS = 24
MIN_OBSERVATION_ACTIONS = 8
SUCCESS_PROTECT_ACTIONS = 18
DWE_STRICT_LOG_COVERAGE = True
NO_IMPACT_STREAK_FOR_POLICY_CHANGE = 3
NO_IMPACT_STREAK_FOR_STOP = 8

# Difference-Weighted Exploitation signals. Positive terms reward causal evidence;
# negative terms penalize wasted trajectories. These are current-game-only signals.
EXPLOIT_WEIGHTS = {
    "score": 1.50,
    "level_complete": 4.00,
    "progress_velocity": 2.25,
    "novel_state": 1.00,
    "causal_confidence": 1.75,
    "target_proximity": 2.50,
    "stall": -2.00,
    "repeat_loop": -3.50,
    "no_progress": -2.75,
    "no_impact": -4.25,
    "terminal_loss": -4.00,
}

# Game weight answers: "is this game worth more global computation?"
# Strategy weight answers: "is the current local behavior worth repeating?"
GAME_WEIGHT_DECAY = 0.94
STRATEGY_WEIGHT_DECAY = 0.88
GAME_WEIGHT_LIMIT = 12.0
STRATEGY_WEIGHT_LIMIT = 12.0

# Combined-weight -> advisory action budget only; it is never a binding stop.
DWE_BUDGET_TIERS = (
    (6.0, 309),   # HARD_EXPLOIT
    (3.0, 260),   # EXPLOIT
    (1.0, 210),   # CAUTIOUS_EXPLOIT
    (-1.0, 160),  # BALANCED
    (-3.0, 120),  # EXPLORE / policy transition
    (-999.0, 84), # probable stop-loss trajectory
)

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_action_cap = getattr(bm.solver, "max_actions_per_game", None)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_actions_per_game = GLOBAL_UNCAPPED_ACTION_LIMIT

# Optional grafts stay within the same current game and current run. A larger
# working context is retained; compact DWE/ADL state prevents raw-history growth.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert bm.solver.max_actions_per_game == GLOBAL_UNCAPPED_ACTION_LIMIT
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"ls20_hard_cap={LS20_MAX_MOVES} other_games_action_cap=NONE "
    f"stall={MAX_STALL_ACTIONS} "
    f"no_progress={MAX_NO_PROGRESS_ACTIONS} "
    f"success_protect={SUCCESS_PROTECT_ACTIONS} "
    f"context={_graft_flags['context_window']} "
    f"seed={CONTROL_SEED} frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')} "
    f"source_per_game_budget={_original_game_budget}",
    flush=True,
)
print("DWE WEIGHTS:", json.dumps(EXPLOIT_WEIGHTS, sort_keys=True), flush=True)


## 7. Closed-loop ADL + Difference-Weighted Exploitation

The model still performs the two-plan `EXPLOIT` vs `EXPLORE` comparison before each action and a post-move ADL update afterward. In addition, this cell installs a **deterministic runtime DWE auditor** at the real `GameAPI` action boundary.

For every committed action the notebook prints:

- `DWE PRE`: prior game/strategy weights, current decision, live budget, stall/no-progress counters.
- `DWE POST`: before/after score and levels, reward, board/state-change evidence, novelty/loop signals, each weighted term, new weights, and the next allocator decision.

The same records are written to `/kaggle/working/dwe_move_events.jsonl`.


In [ ]:
# === CLOSED-LOOP ADL + DIFFERENCE-WEIGHTED EXPLOITATION ===
import asyncio
import contextvars
import hashlib
import inspect
import json
import math
import os
import sys
import threading
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DWE_MOVE_LOG = WORKING_DIR / "dwe_move_events.jsonl"
DWE_SUMMARY_LOG = WORKING_DIR / "dwe_game_summaries.jsonl"

for _path in (DUAL_PATH_POLICY_LOG, POST_MOVE_ADL_LOG, DWE_MOVE_LOG, DWE_SUMMARY_LOG):
    try:
        _path.unlink(missing_ok=True)
    except Exception:
        pass

CLOSED_LOOP_ADL_INSTRUCTION = r"""
ADLDB CLOSED-LOOP DUAL-PATH + DIFFERENCE-WEIGHTED EXPLOITATION POLICY

You have exactly ONE real environment trajectory for the current game. Never
fork, clone, reset for speculation, or use another game's state. Use only
observations/actions/rewards/transitions learned in THIS game in THIS run.

BEFORE EVERY REAL ACTION
1. Construct exactly two legal candidate actions from the same current state:
   A = EXPLOIT: shortest move supported by confirmed causal evidence.
   B = EXPLORE: highest-information legal move not already exhausted.
2. Compare legality, predicted progress, predicted frame/state change,
   information gain, loop risk, action cost, and current-game consistency.
3. Maintain two current-game-only values:
   GAME_EXPLOIT_WEIGHT: whether this GAME deserves more computation.
   STRATEGY_EXPLOIT_WEIGHT: whether the CURRENT STRATEGY deserves repetition.
4. Print/record in your reasoning trace before the tool call:

DWE_PRE_DECISION:
STEP=<integer>
GAME_WEIGHT=<number>
STRATEGY_WEIGHT=<number>
A_ACTION=<candidate A>
B_ACTION=<candidate B>
SELECT=<A or B>
MODE=<HARD_EXPLOIT|EXPLOIT|CAUTIOUS_EXPLOIT|BALANCED|EXPLORE|CHANGE_POLICY>
WHY=<current-game evidence only>

Then issue exactly ONE real environment action.

IMMEDIATELY AFTER EVERY REAL ACTION
Compare pre-state, prediction, action and actual returned state. Record:

POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<number or unknown>
LEVEL_DELTA=<number or unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game lesson>
NEXT_BIAS=<exploit/explore/change_policy/neutral>

Perception/control principles:
- use the full current frame; treat animation/change as evidence, not decoration;
- optimize level depth and verified score progress;
- a visual change confined to a deterministic HUD/moves band is NO_IMPACT;
- ACTION7 is legal when exposed by the environment;

DWE exploitation principles:
- verified level completion is the strongest positive signal;
- positive score/reward/progress increases both game and strategy value;
- novel useful transitions increase information value;
- repeated unchanged states, loops and no-progress streaks reduce strategy value;
- a promising game with a weak strategy means CHANGE_POLICY, not immediate abandon;
- sustained low game/strategy value means CHANGE_POLICY and continued exploration;
- after verified success, exploit the causal pattern for a protected window;
- never let one lucky early transition permanently monopolize the budget.

The runtime independently audits these decisions and prints DWE PRE / DWE POST
for every actual environment action. DWE never terminates a nonterminal game.
Only ls20 has a hard action cap (309). Use latest current-game evidence only.
""".strip()


class ClosedLoopADLToolAgent(ToolAgent):
    """Duck ToolAgent with dual-path ADL and explicit DWE reasoning requirements."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "Qwen/Qwen3.8-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    kwargs = {
        "model": model,
        "timeout": bm.solver.analyzer_timeout,
        "save_request_logs": bm.solver.save_request_logs,
        "base_url": base_url,
        "provider": "vllm",
    }
    # Preserve compatibility with multiple ToolAgent versions while passing the
    # control seed at request level whenever the installed implementation
    # explicitly supports a seed-bearing argument.
    try:
        base_params = inspect.signature(ToolAgent.__init__).parameters
    except Exception:
        base_params = {}
    if "seed" in base_params:
        kwargs["seed"] = CONTROL_SEED
    elif "request_kwargs" in base_params:
        kwargs["request_kwargs"] = {"seed": CONTROL_SEED}
    elif "model_kwargs" in base_params:
        kwargs["model_kwargs"] = {"seed": CONTROL_SEED}
    return ClosedLoopADLToolAgent(**kwargs)


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory


def _clip(value, low, high):
    return max(low, min(high, float(value)))


def _num(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        x = float(value)
        if math.isfinite(x):
            return x
    except Exception:
        pass
    return default


def _read(obj, names, default=None):
    if obj is None:
        return default
    for name in names:
        try:
            if isinstance(obj, Mapping) and name in obj:
                value = obj[name]
            elif hasattr(obj, name):
                value = getattr(obj, name)
            else:
                continue
            if callable(value):
                continue
            if value is not None:
                return value
        except Exception:
            continue
    return default


def _walk_candidates(obj, max_depth=2):
    """Yield a small, safe object graph for score/state field discovery."""
    seen = set()
    queue = deque([(obj, 0)])
    child_names = (
        "state", "game_state", "observation", "result", "info", "metadata",
        "response", "frame", "board", "env", "game", "run",
    )
    while queue:
        cur, depth = queue.popleft()
        if cur is None or id(cur) in seen:
            continue
        seen.add(id(cur))
        yield cur
        if depth >= max_depth:
            continue
        for name in child_names:
            nxt = _read(cur, (name,), None)
            if nxt is not None and not isinstance(nxt, (str, bytes, int, float, bool)):
                queue.append((nxt, depth + 1))


def _find_number(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _num(_read(candidate, names, None), None)
            if value is not None:
                return value
    return default


def _find_bool(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _read(candidate, names, None)
            if isinstance(value, bool):
                return value
            if isinstance(value, (int, float)) and value in (0, 1):
                return bool(value)
            if isinstance(value, str):
                v = value.strip().lower()
                if v in {"true", "yes", "won", "lost", "done", "terminal", "game_over"}:
                    return True
                if v in {"false", "no", "playing", "active", "running"}:
                    return False
    return default


def _extract_game_id(*objs):
    names = ("game_id", "env_name", "environment_id", "game_name", "name")
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            value = _read(candidate, names, None)
            if value is not None:
                text = str(value).strip()
                if text:
                    return text
    return "unknown"


def _extract_action(args, kwargs):
    for key in ("action", "action_name", "action_spec", "move", "command"):
        if key in kwargs:
            return str(kwargs[key])
    for value in args:
        if value is None:
            continue
        text = str(value)
        if text and len(text) <= 300:
            return text
    return "unknown"


def _visual_payload(*objs):
    """Return the first likely 2-D/3-D visual state payload without mutating it."""
    field_names = (
        "board", "grid", "frame", "image", "observation", "pixels",
        "screen", "state_matrix", "board_state",
    )
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=2):
            for name in field_names:
                value = _read(candidate, (name,), None)
                if value is None:
                    continue
                try:
                    if hasattr(value, "tolist"):
                        value = value.tolist()
                    # Require a matrix-like payload; text observations are not
                    # useful for the deterministic HUD-band comparison.
                    if (
                        isinstance(value, (list, tuple))
                        and len(value) >= 3
                        and isinstance(value[0], (list, tuple))
                    ):
                        return value
                except Exception:
                    continue
    return None


def _hash_payload(value):
    if value is None:
        return None
    try:
        payload = json.dumps(
            value,
            sort_keys=True,
            default=str,
            separators=(",", ":"),
        )
    except Exception:
        payload = repr(value)
    if not payload or len(payload) <= 4:
        return None
    return hashlib.sha1(
        payload[:2_000_000].encode("utf-8", "replace")
    ).hexdigest()[:16]


def _core_visual_payload(value):
    """Remove only thin outer HUD/moves bands; retain almost the entire board.

    This is deliberately conservative. The no-impact detector fires only when
    the full frame changes while this core stays identical and there is no
    score/reward/level progress. It therefore cannot manufacture positive
    evidence; it only discounts likely cosmetic/HUD-only changes.
    """
    if value is None or not isinstance(value, (list, tuple)) or len(value) < 8:
        return value
    rows = list(value)
    width = min(
        (len(row) for row in rows if isinstance(row, (list, tuple))),
        default=0,
    )
    if width < 8:
        return value

    # Trim 6.25% from each edge, capped so at least 6x6 content remains.
    trim_y = min(max(1, len(rows) // 16), max(1, (len(rows) - 6) // 2))
    trim_x = min(max(1, width // 16), max(1, (width - 6) // 2))
    core = []
    for row in rows[trim_y:len(rows) - trim_y]:
        if isinstance(row, (list, tuple)):
            core.append(list(row)[trim_x:width - trim_x])
    return core or value


def _stable_signature(*objs):
    return _hash_payload(_visual_payload(*objs))


def _core_signature(*objs):
    visual = _visual_payload(*objs)
    return _hash_payload(_core_visual_payload(visual))

def _snapshot(api, result=None):
    objs = tuple(x for x in (result, api) if x is not None)
    score = _find_number(objs, ("score", "current_score", "total_score", "game_score", "final_score"), None)
    levels = _find_number(objs, ("levels_completed", "level_completed_count", "completed_levels", "level"), None)
    reward = _find_number((result,), ("reward", "score_delta", "delta_reward"), None)
    board_changed = _find_bool((result,), ("board_changed", "state_changed", "frame_changed", "changed"), None)
    level_completed = _find_bool((result,), ("level_completed", "completed_level", "level_won"), None)
    game_over = _find_bool(objs, ("game_over", "done", "terminal", "is_done", "finished"), None)
    won = _find_bool(objs, ("won", "is_won", "victory"), None)
    lost = _find_bool(objs, ("lost", "is_lost", "defeat"), None)
    signature = _stable_signature(result, api)
    core_signature = _core_signature(result, api)
    return {
        "score": score,
        "levels": int(levels) if levels is not None else None,
        "reward": reward,
        "board_changed": board_changed,
        "level_completed": level_completed,
        "game_over": game_over,
        "won": won,
        "lost": lost,
        "signature": signature,
        "core_signature": core_signature,
    }


def _dwe_game_key(game_id):
    value = str(game_id or "").strip().lower()
    return value.split("-", 1)[0] if value else "unknown"

def _is_ls20_game(game_id):
    return _dwe_game_key(game_id) == "ls20"

def _hard_action_cap(game_id):
    return LS20_MAX_MOVES if _is_ls20_game(game_id) else None

def _hard_cap_label(game_id):
    cap = _hard_action_cap(game_id)
    return str(cap) if cap is not None else "UNCAPPED"

@dataclass
class DWEGameState:
    game_id: str
    move: int = 0
    last_score: float = 0.0
    last_levels: int = 0
    game_weight: float = 0.0
    strategy_weight: float = 0.0
    combined_weight: float = 0.0
    progress_velocity: float = 0.0
    stall_streak: int = 0
    no_progress_streak: int = 0
    repeat_streak: int = 0
    no_impact_streak: int = 0
    success_protect_until: int = 0
    live_budget: int = 160
    decision: str = "EXPLORE"
    reason: str = "initial observation"
    last_signature: str | None = None
    last_core_signature: str | None = None
    seen_signatures: deque = field(default_factory=lambda: deque(maxlen=96))
    last_terms: dict = field(default_factory=dict)


class DifferenceWeightedExploitation:
    def __init__(self):
        self._states = {}
        self._lock = threading.RLock()

    def state(self, game_id):
        key = str(game_id or "unknown")
        with self._lock:
            if key not in self._states:
                self._states[key] = DWEGameState(game_id=key)
            return self._states[key]

    @staticmethod
    def _budget(weight):
        for threshold, budget in DWE_BUDGET_TIERS:
            if weight >= threshold:
                return int(min(LS20_MAX_MOVES, budget))
        return int(LS20_MAX_MOVES)

    def pre(self, game_id, action):
        with self._lock:
            st = self.state(game_id)
            print(
                "DWE PRE "
                f"game={st.game_id} move={st.move + 1:03d} action={action} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} mode={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"protect_until={st.success_protect_until} reason={st.reason}",
                flush=True,
            )
            return {
                "move": st.move + 1,
                "score": st.last_score,
                "levels": st.last_levels,
                "signature": st.last_signature,
                "core_signature": st.last_core_signature,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "decision": st.decision,
            }

    def post(self, game_id, action, before, after):
        with self._lock:
            st = self.state(game_id)
            st.move += 1

            before_score = _num(before.get("score"), st.last_score)
            if before_score is None:
                before_score = st.last_score
            after_score = _num(after.get("score"), None)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = before_score + reward
            score_delta = after_score - before_score

            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            level_event = bool(after.get("level_completed"))
            if after_levels is None:
                after_levels = before_levels + (1 if level_event else 0)
            level_delta = max(0, int(after_levels) - int(before_levels))
            level_event = bool(level_event or level_delta > 0)

            sig = after.get("signature")
            prev_sig = st.last_signature
            seen_before = set(st.seen_signatures)
            novel = bool(sig and sig not in seen_before)
            repeated = bool(sig and (sig == prev_sig or sig in seen_before))
            board_changed = after.get("board_changed")
            if board_changed is None and sig and prev_sig:
                board_changed = sig != prev_sig
            if board_changed is None:
                board_changed = bool(score_delta != 0 or level_event or reward != 0)

            core_sig = after.get("core_signature")
            prev_core_sig = st.last_core_signature
            core_changed = None
            if core_sig and prev_core_sig:
                core_changed = core_sig != prev_core_sig

            positive_score = score_delta > 1e-9
            positive_reward = reward > 1e-9
            meaningful_progress = bool(level_event or positive_score or positive_reward)

            # NO_IMPACT = apparent frame/board activity confined to the thin outer
            # band, with no verified reward/score/level progress.
            no_impact = bool(
                board_changed
                and core_changed is False
                and not meaningful_progress
            )
            effective_board_changed = bool(board_changed and not no_impact)
            effective_novel = bool(novel and not no_impact)
            state_activity = bool(effective_board_changed or effective_novel)
            loop_signal = bool(repeated and not meaningful_progress)

            if no_impact:
                st.no_impact_streak += 1
            else:
                st.no_impact_streak = 0

            if meaningful_progress:
                st.no_progress_streak = 0
            else:
                st.no_progress_streak += 1

            if meaningful_progress or state_activity:
                st.stall_streak = 0
            else:
                st.stall_streak += 1

            if loop_signal:
                st.repeat_streak += 1
            else:
                st.repeat_streak = 0

            # Progress value is deliberately conservative: visual novelty alone is useful
            # information, but weaker than verified score/reward/level progress.
            progress_value = 0.0
            if level_event:
                progress_value += 1.00
            if positive_score:
                progress_value += min(0.65, 0.20 + abs(score_delta))
            if positive_reward:
                progress_value += min(0.40, 0.10 + abs(reward))
            if effective_board_changed:
                progress_value += 0.12
            if effective_novel:
                progress_value += 0.10
            if loop_signal:
                progress_value -= 0.30
            if no_impact:
                progress_value -= 0.25
            if not meaningful_progress and not state_activity:
                progress_value -= 0.15
            progress_value = _clip(progress_value, -1.0, 1.0)
            st.progress_velocity = _clip(0.70 * st.progress_velocity + 0.30 * progress_value, -1.0, 1.0)

            # Causal confidence is tied only to observed current-game consequences.
            causal_confidence = 0.0
            if level_event:
                causal_confidence = 1.0
            elif positive_score:
                causal_confidence = 0.85
            elif positive_reward:
                causal_confidence = 0.70
            elif effective_board_changed and effective_novel:
                causal_confidence = 0.35
            elif effective_board_changed:
                causal_confidence = 0.20

            normalized_score = _clip(after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 2.0)
            target_proximity = 1.0 if after_score >= TARGET_MIN_GAME_SCORE else _clip(
                after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 1.0
            )
            # A stagnant near-target game should not monopolize budget, so proximity is
            # gated by recent causal progress velocity.
            target_activity_gate = 0.20 + 0.80 * max(0.0, st.progress_velocity)
            target_signal = target_proximity * target_activity_gate
            stall_ratio = _clip(st.stall_streak / max(MAX_STALL_ACTIONS, 1), 0.0, 1.0)
            no_progress_ratio = _clip(st.no_progress_streak / max(MAX_NO_PROGRESS_ACTIONS, 1), 0.0, 1.0)
            repeat_ratio = _clip(st.repeat_streak / 4.0, 0.0, 1.0)
            no_impact_ratio = _clip(
                st.no_impact_streak / max(NO_IMPACT_STREAK_FOR_STOP, 1),
                0.0,
                1.0,
            )
            terminal_loss = bool(after.get("lost") or (after.get("game_over") and not after.get("won")))

            terms = {
                "score": EXPLOIT_WEIGHTS["score"] * normalized_score,
                "level_complete": EXPLOIT_WEIGHTS["level_complete"] * (1.0 if level_event else 0.0),
                "progress_velocity": EXPLOIT_WEIGHTS["progress_velocity"] * st.progress_velocity,
                "novel_state": EXPLOIT_WEIGHTS["novel_state"] * (1.0 if effective_novel else 0.0),
                "causal_confidence": EXPLOIT_WEIGHTS["causal_confidence"] * causal_confidence,
                "target_proximity": EXPLOIT_WEIGHTS["target_proximity"] * target_signal,
                "stall": EXPLOIT_WEIGHTS["stall"] * stall_ratio,
                "repeat_loop": EXPLOIT_WEIGHTS["repeat_loop"] * repeat_ratio,
                "no_progress": EXPLOIT_WEIGHTS["no_progress"] * no_progress_ratio,
                "no_impact": EXPLOIT_WEIGHTS["no_impact"] * no_impact_ratio,
                "terminal_loss": EXPLOIT_WEIGHTS["terminal_loss"] * (1.0 if terminal_loss else 0.0),
            }

            game_signal = sum(terms.values())
            # Strategy weight emphasizes immediate causal evidence and punishes local
            # failure more strongly than game weight, allowing CHANGE_POLICY on good games.
            strategy_signal = (
                5.00 * (1.0 if level_event else 0.0)
                + 2.75 * st.progress_velocity
                + 2.00 * causal_confidence
                + 0.50 * (1.0 if effective_novel else 0.0)
                - 2.50 * stall_ratio
                - 4.00 * repeat_ratio
                - 3.25 * no_progress_ratio
                - 4.25 * no_impact_ratio
                - 4.50 * (1.0 if terminal_loss else 0.0)
            )

            st.game_weight = _clip(
                GAME_WEIGHT_DECAY * st.game_weight + game_signal,
                -GAME_WEIGHT_LIMIT,
                GAME_WEIGHT_LIMIT,
            )
            st.strategy_weight = _clip(
                STRATEGY_WEIGHT_DECAY * st.strategy_weight + strategy_signal,
                -STRATEGY_WEIGHT_LIMIT,
                STRATEGY_WEIGHT_LIMIT,
            )
            st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight

            if level_event or positive_score or positive_reward:
                st.success_protect_until = max(
                    st.success_protect_until,
                    st.move + SUCCESS_PROTECT_ACTIONS,
                )

            st.live_budget = self._budget(st.combined_weight)
            if st.move <= st.success_protect_until:
                st.live_budget = max(st.live_budget, min(LS20_MAX_MOVES, st.success_protect_until + 12))

            protected = st.move <= st.success_protect_until
            if after.get("game_over"):
                st.decision = "TERMINAL"
                st.reason = "environment reported terminal state"
            elif st.move < MIN_OBSERVATION_ACTIONS:
                st.decision = "EXPLORE"
                st.reason = "minimum observation window"
            elif level_event and st.combined_weight >= 3.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "verified level completion"
            elif meaningful_progress and st.combined_weight >= 1.0:
                st.decision = "EXPLOIT"
                st.reason = "verified current-game progress"
            elif protected:
                st.decision = "EXPLOIT_PROTECTED"
                st.reason = "recent success protected exploit window"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_STOP and st.game_weight < 1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact with low game value; change strategy and continue"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact actions; abandon current local strategy"
            elif st.no_progress_streak >= MAX_NO_PROGRESS_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "promising game but current strategy has no progress"
                else:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "sustained no progress with low game value; change strategy and continue"
            elif st.stall_streak >= MAX_STALL_ACTIONS:
                if st.game_weight >= 1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall detected in a still-promising game"
                elif st.combined_weight < -1.0:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "hard stall plus negative evidence; change strategy and continue"
                else:
                    st.decision = "CHANGE_POLICY"
                    st.reason = "stall threshold reached"
            elif st.game_weight >= 1.0 and st.strategy_weight <= -1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "game weight high while strategy weight is low"
            elif st.combined_weight >= 6.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "very high combined exploit weight"
            elif st.combined_weight >= 3.0:
                st.decision = "EXPLOIT"
                st.reason = "high combined exploit weight"
            elif st.combined_weight >= 1.0:
                st.decision = "CAUTIOUS_EXPLOIT"
                st.reason = "positive combined exploit weight"
            elif st.combined_weight >= -1.0:
                st.decision = "BALANCED"
                st.reason = "mixed current-game evidence"
            else:
                st.decision = "EXPLORE"
                st.reason = "low exploit confidence; seek information"

            st.last_score = float(after_score)
            st.last_levels = int(after_levels)
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            if core_sig:
                st.last_core_signature = core_sig
            st.last_terms = {k: round(v, 6) for k, v in terms.items()}

            event = {
                "game_id": st.game_id,
                "move": st.move,
                "action": action,
                "before_score": before_score,
                "after_score": after_score,
                "score_delta": score_delta,
                "before_levels": before_levels,
                "after_levels": after_levels,
                "level_delta": level_delta,
                "reward": reward,
                "board_changed": bool(board_changed),
                "effective_board_changed": bool(effective_board_changed),
                "core_changed": core_changed,
                "no_impact": bool(no_impact),
                "novel_state": novel,
                "effective_novel_state": effective_novel,
                "loop_signal": loop_signal,
                "progress_value": progress_value,
                "progress_velocity": st.progress_velocity,
                "causal_confidence": causal_confidence,
                "target_proximity": target_proximity,
                "stall_streak": st.stall_streak,
                "no_progress_streak": st.no_progress_streak,
                "repeat_streak": st.repeat_streak,
                "no_impact_streak": st.no_impact_streak,
                "terms": st.last_terms,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "combined_weight": st.combined_weight,
                "decision": st.decision,
                "reason": st.reason,
                "live_budget": st.live_budget,
                "hard_budget": _hard_action_cap(st.game_id),
                "action_cap_policy": "ls20=309; all other games uncapped",
                "live_budget_binding": False,
                "success_protect_until": st.success_protect_until,
                "game_over": bool(after.get("game_over")),
                "won": bool(after.get("won")),
                "lost": bool(after.get("lost")),
            }
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")

            term_text = ",".join(f"{k}={v:+.2f}" for k, v in st.last_terms.items())
            print(
                "DWE POST "
                f"game={st.game_id} move={st.move:03d} action={action} "
                f"score={before_score:.6f}->{after_score:.6f} dscore={score_delta:+.6f} "
                f"levels={before_levels}->{after_levels} dlevel={level_delta:+d} reward={reward:+.4f} "
                f"changed={int(bool(board_changed))} effective_changed={int(bool(effective_board_changed))} "
                f"NO_IMPACT={int(bool(no_impact))} novel={int(novel)} loop={int(loop_signal)} "
                f"progress={progress_value:+.3f} velocity={st.progress_velocity:+.3f} "
                f"target={target_proximity:.3f} stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} next={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"protect_until={st.success_protect_until} "
                f"terms=[{term_text}] reason={st.reason}",
                flush=True,
            )
            return event

    def should_stop(self, game_id):
        """Binding DWE stop: only ls20 at 309 actions."""
        with self._lock:
            st = self.state(game_id)
            cap = _hard_action_cap(st.game_id)
            if cap is not None and st.move >= cap:
                return True, f"ls20 hard action ceiling {cap}"
            return False, "DWE action-uncapped"

    def summaries(self):
        with self._lock:
            return [
                {
                    "game_id": st.game_id,
                    "moves": st.move,
                    "score": st.last_score,
                    "levels": st.last_levels,
                    "game_weight": st.game_weight,
                    "strategy_weight": st.strategy_weight,
                    "combined_weight": st.combined_weight,
                    "decision": st.decision,
                    "reason": st.reason,
                    "live_budget": st.live_budget,
                    "stall_streak": st.stall_streak,
                    "no_progress_streak": st.no_progress_streak,
                    "repeat_streak": st.repeat_streak,
                    "no_impact_streak": st.no_impact_streak,
                    "hard_action_cap": _hard_action_cap(st.game_id),
                    "live_budget_binding": False,
                }
                for st in self._states.values()
            ]


DWE_ALLOCATOR = DifferenceWeightedExploitation()
_DWE_ACTION_DEPTH = contextvars.ContextVar("dwe_action_depth", default=0)
_DWE_PATCHED_ACTION_METHODS = []
_DWE_PATCHED_STOP_METHODS = []


def _method_action_score(name, method):
    score = 0
    lname = name.lower()
    if lname in {"step", "act", "action", "execute_action", "perform_action", "take_action", "play_action", "apply_action"}:
        score += 8
    if "action" in lname:
        score += 4
    if any(token in lname for token in ("step", "move", "act", "play")):
        score += 2
    try:
        sig = inspect.signature(method)
        params = {p.lower() for p in sig.parameters}
        if params.intersection({"action", "action_name", "action_spec", "move", "command"}):
            score += 8
    except Exception:
        pass
    try:
        source = inspect.getsource(method).lower()
        if "board_changed" in source or "level_completed" in source or ".step(" in source:
            score += 4
    except Exception:
        pass
    return score


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            DWE_ALLOCATOR.post(game_id, action, before, after)
            return result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_gameapi_action_hook(game_apis):
    classes = []
    for api in game_apis:
        if api.__class__ not in classes:
            classes.append(api.__class__)
    installed = []
    for cls in classes:
        candidates = []
        for name in dir(cls):
            if name.startswith("_"):
                continue
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if not callable(method):
                continue
            score = _method_action_score(name, method)
            if score > 0:
                candidates.append((score, name))
        candidates.sort(reverse=True)
        if not candidates:
            raise RuntimeError(
                f"DWE could not identify a real action method on {cls.__module__}.{cls.__name__}; "
                "refusing to run without per-move exploit logging."
            )
        best_score, best_name = candidates[0]
        if best_score < 6:
            raise RuntimeError(
                f"DWE action-boundary confidence too low for {cls.__name__}: {candidates[:8]}"
            )
        if _wrap_action_method(cls, best_name):
            installed.append(f"{cls.__module__}.{cls.__name__}.{best_name}")
        print(
            f"DWE ACTION HOOK class={cls.__module__}.{cls.__name__} "
            f"method={best_name} confidence={best_score} candidates={candidates[:6]}",
            flush=True,
        )
    return installed


def _object_game_id(self, args, kwargs):
    return _extract_game_id(self, *args, *kwargs.values())


def _wrap_should_stop(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_stop_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            original_result = await original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result
    else:
        def wrapped(self, *args, **kwargs):
            original_result = original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_stop_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_STOP_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_should_stop_hooks(game_apis):
    classes = {api.__class__ for api in game_apis}
    # The stop predicate can live on GameAPI, a solver/session class, or a run class.
    for module_name, module in list(sys.modules.items()):
        if not module or not (
            module_name.startswith("taaf") or module_name.startswith("inference")
        ):
            continue
        try:
            values = list(vars(module).values())
        except Exception:
            continue
        for obj in values:
            if inspect.isclass(obj):
                classes.add(obj)

    installed = []
    for cls in classes:
        for name in ("should_stop", "_should_stop"):
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if callable(method) and _wrap_should_stop(cls, name):
                full = f"{cls.__module__}.{cls.__name__}.{name}"
                installed.append(full)
                print(f"DWE STOP HOOK {full}", flush=True)
    if not installed:
        print(
            "DWE STOP HOOK WARNING: no should_stop predicate found; hard ceiling remains active. "
            "Per-move DWE logging and policy weighting are still active.",
            flush=True,
        )
    return installed


def _install_dwe_runtime_hooks(game_apis):
    action_hooks = _install_gameapi_action_hook(game_apis)
    stop_hooks = _install_should_stop_hooks(game_apis)
    if not action_hooks and not _DWE_PATCHED_ACTION_METHODS:
        raise RuntimeError("DWE requires an action-boundary hook; none was installed.")
    if not stop_hooks and not _DWE_PATCHED_STOP_METHODS:
        raise RuntimeError("DWE requires a should_stop hook to enforce ls20=309; none was installed.")
    print(
        "DWE RUNTIME ACTIVE "
        f"action_hooks={len(action_hooks) or len(_DWE_PATCHED_ACTION_METHODS)} "
        f"stop_hooks={len(stop_hooks) or len(_DWE_PATCHED_STOP_METHODS)} "
        "log_every_move=True",
        flush=True,
    )


print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("DWE ACTIVE: separate game and strategy exploit weights", flush=True)
print("DWE LOGGING: PRE + POST for every real environment action", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME MEMORY: DISABLED", flush=True)

print("ACTION CAP POLICY: ls20=309; every other game action-UNCAPPED", flush=True)


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===

if BUILD_ID != "ADLDB-Q38-FINAL-TESTED-v3-20260822":
    raise RuntimeError(
        f"STALE NOTEBOOK DETECTED AT GAMEPLAY BOUNDARY: {BUILD_ID!r}"
    )
print(f"GAMEPLAY BUILD VERIFIED: {BUILD_ID}", flush=True)

# === FINAL TESTED-v3 INVARIANTS ===
assert BUILD_ID == "ADLDB-Q38-FINAL-TESTED-v3-20260822"
assert ANALYZER_MODEL_ID == "Qwen/Qwen3.8-27B-FP8"
assert QWEN_MODEL_DIR.is_dir()
assert (QWEN_MODEL_DIR / "config.json").is_file()
assert (WORKING_DIR / "all_requirements_audit.json").is_file()
assert _hard_action_cap("ls20-test") == 309
assert _hard_action_cap("vc33-test") is None
assert bm.solver.max_actions_per_game == GLOBAL_UNCAPPED_ACTION_LIMIT

_final_req_audit = json.loads(
    (WORKING_DIR / "all_requirements_audit.json").read_text(encoding="utf-8")
)
if _final_req_audit.get("wheelhouse_lock_failures"):
    raise RuntimeError("FINAL INVARIANT FAILED: vLLM lock reconciliation")
if _final_req_audit.get("source_requirements_unresolved"):
    raise RuntimeError(
        "FINAL INVARIANT FAILED: unresolved source requirements "
        f"{_final_req_audit['source_requirements_unresolved']}"
    )

print(
    "FINAL TESTED-v3 INVARIANTS PASS "
    f"build={BUILD_ID} model={ANALYZER_MODEL_ID} "
    "inputs=4 requirements=ready ls20=309 others=uncapped",
    flush=True,
)

# === FINAL STARTUP GATE BEFORE ANY ARC ENVIRONMENT ===
_INPUT_MODEL_READY_PATH = WORKING_DIR / "INPUT_MODEL_READY.json"
if not _INPUT_MODEL_READY_PATH.is_file():
    raise RuntimeError(
        "INPUT_MODEL_READY.json missing. "
        "All inputs/model were not fully loaded."
    )

_input_model_ready = json.loads(
    _INPUT_MODEL_READY_PATH.read_text(encoding="utf-8")
)

_required_ready_flags = [
    "all_static_inputs_loaded",
    "competition_loaded",
    "taaf_source_loaded",
    "vllm_wheelhouse_loaded",
    "all_requirements_loaded",
    "all_source_trees_loaded",
    "qwen38_snapshot_loaded",
    "vllm_server_started",
    "chat_completion_smoke_passed",
]

_failed_ready_flags = [
    flag
    for flag in _required_ready_flags
    if _input_model_ready.get(flag) is not True
]
if _failed_ready_flags:
    raise RuntimeError(
        "INPUT/MODEL STARTUP GATE FAILED FLAGS: "
        + ", ".join(_failed_ready_flags)
    )

if _input_model_ready.get("served_model_id") != ANALYZER_MODEL_ID:
    raise RuntimeError(
        "Wrong model at gameplay boundary: "
        f"{_input_model_ready.get('served_model_id')}"
    )

print(
    "FINAL STARTUP GATE PASS — "
    "ALL INPUTS + ALL REQUIREMENTS + QWEN3.8 MODEL LOADED",
    flush=True,
)

# === STAGE 7A: REQUIREMENTS GUARD BEFORE GAME ENVIRONMENTS ===
_REQUIREMENTS_AUDIT_PATH = WORKING_DIR / "all_requirements_audit.json"
if not _REQUIREMENTS_AUDIT_PATH.is_file():
    raise RuntimeError(
        "all_requirements_audit.json is missing; requirements stage did not complete."
    )

_requirements_audit = json.loads(
    _REQUIREMENTS_AUDIT_PATH.read_text(encoding="utf-8")
)
if _requirements_audit.get("wheelhouse_lock_failures"):
    raise RuntimeError(
        "Pinned wheelhouse requirements are not fully reconciled."
    )

_expected_imports = {
    "arc_agi",
    "numpy",
    "pandas",
    "pyarrow",
    "torch",
    "packaging",
    "inference.agent.action_names",
    "inference.framework.solver",
    "inference.agent.tool_agent",
    "taaf.game_api",
}
_loaded_imports = set(
    (_requirements_audit.get("required_imports") or {}).keys()
)
if not _expected_imports.issubset(_loaded_imports):
    raise RuntimeError(
        "Required import audit incomplete: "
        f"missing={sorted(_expected_imports - _loaded_imports)}"
    )

print(
    "PIPELINE STAGE 7A — REQUIREMENTS GUARD PASS "
    f"lock_entries={_requirements_audit.get('wheelhouse_lock_entries')} "
    f"imports={len(_loaded_imports)}",
    flush=True,
)

import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(os.environ["ARC_AGI3_ENVIRONMENTS_DIR"])
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# Validate per-game action-cap mapping before gameplay.
_cap_contract = {_extract_game_id(api): _hard_action_cap(_extract_game_id(api)) for api in game_apis}
_bad_non_ls20 = [(gid, cap) for gid, cap in _cap_contract.items() if not _is_ls20_game(gid) and cap is not None]
if _bad_non_ls20:
    raise RuntimeError(f"Non-ls20 games unexpectedly capped: {_bad_non_ls20}")
_ls20_caps = [(gid, cap) for gid, cap in _cap_contract.items() if _is_ls20_game(gid)]
if _ls20_caps and any(cap != LS20_MAX_MOVES for _, cap in _ls20_caps):
    raise RuntimeError(f"ls20 cap contract violated: {_ls20_caps}")
print(f"PIPELINE STAGE 8 — ACTION CAP CONTRACT ls20={LS20_MAX_MOVES} all_other_games=UNCAPPED", flush=True)

# Install deterministic per-move exploit logging and stop-loss hooks only after
# the actual GameAPI class(es) have been constructed. The notebook refuses to
# start the benchmark if it cannot identify the action boundary confidently.
_install_dwe_runtime_hooks(game_apis)

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
# Wall-clock safety remains independent of action caps. Non-ls20 games are action-uncapped, not time-unlimited.
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "ls20_max_moves": LS20_MAX_MOVES,
    "action_cap_policy": {"ls20": LS20_MAX_MOVES, "all_other_games": None},
    "global_action_limit_sentinel": GLOBAL_UNCAPPED_ACTION_LIMIT,
    "dwe_stop_loss_binding": False,
    "dwe_live_budget_binding": False,
    "control_seed": CONTROL_SEED,
    "analyzer_model": os.environ.get("INFERENCE_ANALYZER_MODEL"),
    "qwen_model_dir": str(QWEN_MODEL_DIR),
    "resolved_model_dataset": str(QWEN_MODEL_DIR),
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
    "frame_mode": os.environ.get("ARC3_FRAME_MODE"),
    "state_graph": os.environ.get("ARC3_STATE_GRAPH"),
    "no_impact_weight": EXPLOIT_WEIGHTS["no_impact"],
    "no_impact_policy_change": NO_IMPACT_STREAK_FOR_POLICY_CHANGE,
    "no_impact_stop": NO_IMPACT_STREAK_FOR_STOP,
    "schema": "adldb.arc3.dwe.qwen38.seed20260819.v3",
    "dwe_enabled": True,
    "dwe_log_every_move": True,
    "dwe_weights": EXPLOIT_WEIGHTS,
    "dwe_budget_tiers": [list(x) for x in DWE_BUDGET_TIERS],
    "success_protect_actions": SUCCESS_PROTECT_ACTIONS,
    "min_observation_actions": MIN_OBSERVATION_ACTIONS,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
    "target_score_games": TARGET_SCORE_GAMES,
    "target_min_game_score": TARGET_MIN_GAME_SCORE,
    "max_stall_actions": MAX_STALL_ACTIONS,
    "max_no_progress_actions": MAX_NO_PROGRESS_ACTIONS,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "QWEN38 ADLDB-DWE RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f} "
    f"action_caps=ls20:{LS20_MAX_MOVES},others:UNCAPPED "
    f"dwe_stop_loss_binding=off dwe_live_budget_binding=off "
    f"dwe=on log_every_move=on "
    f"seed={CONTROL_SEED} frame=full model={os.environ.get('INFERENCE_ANALYZER_MODEL')}",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "ADLDB-DWE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


## 10. Final ADLDB/DWE run summary

Summarize scored outcomes and the final exploit state of every game. The output keeps the score metrics needed for the next ADL comparison and writes a compact per-game DWE summary artifact.


In [ ]:
# === FINAL ADLDB / DWE SUMMARY ===
import statistics

runs = list(getattr(bm, "game_runs", []) or [])
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]
positive = sum(score > 0 for score in scores)
at_target = sum(score >= TARGET_MIN_GAME_SCORE for score in scores)

print(
    "QWEN38 ADLDB SUMMARY "
    f"games={len(runs)} "
    f"mean_score={(sum(scores) / len(scores) if scores else 0.0):.6f} "
    f"score_sum={sum(scores):.6f} "
    f"positive_games={positive} "
    f"target_games={at_target}/{TARGET_SCORE_GAMES} "
    f"levels={sum(levels)} "
    f"actions={sum(actions)} "
    f"seed={CONTROL_SEED} frame={os.environ.get('ARC3_FRAME_MODE')} "
    f"model={os.environ.get('INFERENCE_ANALYZER_MODEL')} "
    f"action_caps=ls20:{LS20_MAX_MOVES},others:UNCAPPED",
    flush=True,
)

summaries = DWE_ALLOCATOR.summaries()
with DWE_SUMMARY_LOG.open("w", encoding="utf-8") as fh:
    for item in sorted(summaries, key=lambda x: x["game_id"]):
        fh.write(json.dumps(item, sort_keys=True) + "\n")
        print(
            "DWE GAME SUMMARY "
            f"game={item['game_id']} moves={item['moves']} score={item['score']:.6f} "
            f"levels={item['levels']} gameW={item['game_weight']:+.3f} "
            f"strategyW={item['strategy_weight']:+.3f} combined={item['combined_weight']:+.3f} "
            f"decision={item['decision']} live_budget={item['live_budget']} "
            f"stall={item['stall_streak']} no_progress={item['no_progress_streak']} "
            f"repeat={item['repeat_streak']} "
            f"no_impact={item.get('no_impact_streak', 0)} reason={item['reason']}",
            flush=True,
        )

print(f"DWE MOVE LOG: {DWE_MOVE_LOG}", flush=True)
print(f"DWE SUMMARY LOG: {DWE_SUMMARY_LOG}", flush=True)


## 11. Per-move exploit/ADL audit

Verify that the deterministic DWE action-boundary logger produced one unique post-action exploit record for every recorded game action. In local validation mode, incomplete coverage is a hard failure. In official competition reruns it is surfaced prominently without destroying an otherwise valid submission artifact.


In [ ]:
# === PER-MOVE DWE / POST-MOVE ADL AUDIT ===
from collections import Counter

records = []
if DWE_MOVE_LOG.exists():
    for raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("DWE AUDIT malformed line:", raw[:240], flush=True)

unique_move_keys = {
    (str(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
recorded_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
logged_moves = len(unique_move_keys)
coverage = logged_moves / recorded_actions if recorded_actions else 1.0

modes = Counter(str(item.get("decision", "unknown")) for item in records)
stop_loss = modes.get("STOP_LOSS", 0)
no_impact_moves = sum(1 for item in records if item.get("no_impact"))
change_policy = modes.get("CHANGE_POLICY", 0)
exploit_moves = sum(
    count for mode, count in modes.items()
    if "EXPLOIT" in mode
)

print(
    "DWE MOVE AUDIT "
    f"recorded_actions={recorded_actions} "
    f"unique_logged_moves={logged_moves} "
    f"coverage={coverage:.3f} "
    f"exploit_updates={exploit_moves} "
    f"change_policy_updates={change_policy} "
    f"stop_loss_updates={stop_loss} "
    f"no_impact_moves={no_impact_moves} "
    f"modes={dict(sorted(modes.items()))}",
    flush=True,
)

# Per-game coverage makes any missing trace immediately visible in notebook logs.
logged_by_game = Counter(str(item.get("game_id", "unknown")) for item in records)
for run in getattr(bm, "game_runs", []) or []:
    gid = _game_key(run.game_id)
    # Match either exact full id or normalized game key.
    logged = sum(
        count for key, count in logged_by_game.items()
        if _game_key(key) == gid
    )
    expected = _run_actions(run)
    game_cov = logged / expected if expected else 1.0
    print(
        "DWE GAME AUDIT "
        f"game={gid} actions={expected} logged={logged} coverage={game_cov:.3f}",
        flush=True,
    )

if coverage < 0.999999:
    message = (
        "DWE per-move exploit logging coverage is incomplete: "
        f"{logged_moves}/{recorded_actions} ({coverage:.3%})."
    )
    if DWE_STRICT_LOG_COVERAGE and not TRUE_SUBMISSION:
        raise RuntimeError(message)
    print("WARNING:", message, flush=True)
else:
    print("DWE AUDIT PASS: exploit logic logged for every recorded move", flush=True)

if stop_loss:
    raise RuntimeError(f"Action-uncapped contract violated: {stop_loss} STOP_LOSS decisions logged.")
print("ACTION CAP AUDIT PASS: no DWE STOP_LOSS decisions; only ls20=309 is binding", flush=True)
